# PI-GNN — Real-Data Pipeline Test

Runs the PI-GNN pipeline (`generate_synthetic_growth_data.py`, `anatomy_and_graph_conversion.py`, `pignn_glioma.py`) against real public datasets, on Kaggle (GPU + internet, unlike the offline sandbox the pipeline was originally drafted in).

**Datasets used, and what each one tests:**

| Dataset | Role here | Source |
|---|---|---|
| **RHUH-GBM** (2D slice mirror) | Fast sanity check: real tumor segmentation → supervoxel graph, in 2D | Hugging Face `MedOtter/RHUH-GBM` |
| **LUMIERE** | Real longitudinal baseline → follow-up pair → real `c0`/`c_followup` graph (external-validation-style check, no synthetic step needed) | Kaggle `rodschermitg/lumiere-dataset` |

**UPenn-GBM was dropped from this notebook.** The Kaggle mirror `sauravilalge/upenn-gbm` turned out to be TCIA's raw per-visit clinical DICOM archive (folders named by scanner series description, e.g. `t2Flairaxial`, `ep2ddiffMDDWIPAT`, all `.dcm` files) -- not the curated, co-registered NIfTI release (`images_structural`/`images_segm`/`images_DTI`) that the paper's Method section assumes, which has ground-truth segmentation and DTI-derived FA/AD maps. Without a segmentation, there's no tissue mask to seed a real-anatomy FEM run, and without FA/AD there's no real diffusion tensor -- so this mirror can't close the "real DTI loading" TODO. That gap stays open pending either a Kaggle mirror of the curated NIfTI release or a direct TCIA download.

**Before running on Kaggle:**
1. Create a new Kaggle Notebook, enable a **GPU** accelerator and **Internet** access (Settings panel).
2. Click **+ Add Input** and attach the Kaggle dataset `rodschermitg/lumiere-dataset`. RHUH-GBM is pulled directly from the Hugging Face Hub in Section 2 (no attachment needed).
3. Run cells top to bottom. **Section 3** prints the actual folder/file listing for the attached dataset -- if the LUMIERE loader in Section 6 can't find a file, look at that printout first and adjust the `glob` patterns in the corresponding `find_*` helper. Dataset mirrors get reorganized between versions, so the discovery cell is deliberately defensive rather than hard-coded.

**Honesty note, carried over from the paper's Limitations section:** the RHUH-GBM mirror used here is a 120-row 2D-slice preview (not the full 3D TCIA archive with raw DWI), and neither it nor LUMIERE include diffusion tensor imaging. `load_patient_anatomy_real()` (which fits a real tensor from raw DWI via `dipy`) does not run anywhere in this notebook as a result -- both real-data tests below use an isotropic placeholder tensor where a diffusion tensor is needed at all, which is fine for exercising the graph-construction and solver code but not representative of the anisotropic model described in the paper.


## 1. Environment setup

Kaggle notebooks ship with `torch` and most scientific-Python packages preinstalled. This cell installs the pieces that usually aren't there yet, and is safe to re-run.

In [ ]:
# torch_geometric's wheel must match the preinstalled torch/CUDA build, so install it
# via the official index rather than plain PyPI.
#
# IMPORTANT: none of nibabel/dipy/scikit-image/datasets/huggingface_hub/torch_geometric
# should need torch upgraded -- but torch_geometric's pip metadata has been known to pin
# a torch floor that makes pip's resolver silently swap out the preinstalled torch build
# for a newer one, which on Kaggle can DROP support for older GPUs (e.g. the P100's
# CUDA capability 6.0 -- confirmed to break this way on a real run: "no kernel image is
# available for execution on the device" after Section 1 ran). Pinning the already-
# installed torch version on every install call below prevents pip from ever touching
# it: pip sees the pin is already satisfied and leaves it alone, and if some other
# package genuinely can't coexist with that pinned torch, pip fails loudly here instead
# of silently breaking the GPU three sections later.
import subprocess, sys, torch

PINNED_TORCH = f"torch=={torch.__version__}"
print(f"Pinning {PINNED_TORCH} for every install below (prevents an unwanted torch upgrade).")

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs, PINNED_TORCH], check=False)

torch_version = torch.__version__.split("+")[0]
cuda_tag = "cu121" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  (capability {torch.cuda.get_device_capability(0)})")

pip_install("nibabel", "dipy", "scikit-image", "datasets", "huggingface_hub")
try:
    import torch_geometric  # noqa
    print("torch_geometric already installed:", torch_geometric.__version__)
except ImportError:
    pip_install("torch_geometric")
    try:
        import torch_geometric  # noqa
        print("torch_geometric installed:", torch_geometric.__version__)
    except ImportError:
        print("torch_geometric install failed \u2014 pinned-wheel install as a fallback:")
        pip_install("torch_geometric", "-f",
                    f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html")

# NOTE: deliberately not re-checking torch's version via importlib.reload() here --
# reload() re-executes torch's C++ library registration code (e.g. the "triton"
# TORCH_LIBRARY namespace), which torch does not allow to run twice in one process
# and crashes with "Only a single TORCH_LIBRARY can be used to register the
# namespace triton" (confirmed on a real run). Reloading complex C-extension
# packages is unsafe in general, not just for torch. The PINNED_TORCH constraint
# above already stops pip from touching the installed torch build in the first
# place, and DEVICE is forced to "cpu" in the next cell regardless, so the GPU
# compatibility failure this was guarding against can't happen here anyway.
print(f"torch {torch.__version__} (pinned during installs above; DEVICE is forced to cpu next cell "
      f"so a torch version change here would not reintroduce the earlier GPU issue).")


In [ ]:
import os, glob, json, time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# Forced to CPU on purpose: every graph in this notebook is tiny (145-300 nodes --
# see Sections 4-5), and Section 7 already measured full forward passes at ~5ms on
# CPU. GPU buys nothing here and has just cost real debugging time (a P100
# compatibility break after Section 1's installs, "no kernel image is available
# for execution on the device") for zero benefit. If a future run needs GPU for a
# genuinely larger job, change this back to the commented line below -- but do
# that deliberately, not by default.
DEVICE = "cpu"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

KAGGLE_INPUT = "/kaggle/input"
IS_KAGGLE = os.path.isdir(KAGGLE_INPUT)
print("Running on Kaggle:", IS_KAGGLE)


## 2. Pipeline source (unmodified from the repo)

Pasted as-is from `generate_synthetic_growth_data.py`, `anatomy_and_graph_conversion.py`, and `pignn_glioma.py` so this notebook is self-contained. If you'd rather import them as modules, upload the three files as a Kaggle Dataset/Utility Script and replace these three cells with `from generate_synthetic_growth_data import *` etc.

### 2a. `generate_synthetic_growth_data.py`

In [ ]:
"""
Synthetic data generator for PI-GNN pretraining.

Generates Fisher-KPP reaction-diffusion tumor growth trajectories over
REAL patient anatomy (from UCSF-PDGM / UPenn-GBM segmentations + DTI),
with RANDOMIZED (D, rho) parameters and randomized seed location, so the
PI-GNN can pretrain on a large synthetic set before fine-tuning on the
small amount of real longitudinal data (RHUH-GBM, LUMIERE).

Equation:
    dc/dt = div(D(x) grad c) + rho * c * (1 - c)

Solved here with an explicit finite-volume scheme on a regular voxel grid
(simplest correct discretization; swap in a real FEM mesh solver, e.g.
FEniCS, if you need irregular tetrahedral meshes matching your GNN graph
exactly). Anisotropy comes from a diffusion TENSOR per voxel derived from
DTI, not a scalar -- this preserves the fiber-guided invasion pattern that
matters clinically.

Output format: one .npz per synthetic patient with the full trajectory,
diffusion tensor field, tissue mask, and ground-truth (D_scale, rho) --
directly usable to build the torch_geometric graphs referenced in
pignn_glioma.py (convert grid -> supervoxel graph as a separate step).

Dependencies: numpy, scipy, nibabel
"""

import numpy as np
from scipy.ndimage import gaussian_filter
import os

try:
    import nibabel as nib  # only needed by load_patient_anatomy() with real data
except ImportError:
    nib = None


# ---------------------------------------------------------------------------
# 1. Load real anatomy (tissue mask + DTI tensor field) -- TODO: fill in
# ---------------------------------------------------------------------------
def load_patient_anatomy(patient_dir: str):
    """
    Load a real patient's tissue segmentation and DTI-derived diffusion
    tensor field to use as the anatomical substrate for a SYNTHETIC growth
    simulation (only the growth trajectory is synthetic; the anatomy is real).

    Expected outputs:
        tissue_mask: (X, Y, Z) int array, e.g. 0=background/CSF, 1=gray matter,
                     2=white matter, 3=ventricle
        D_tensor:    (X, Y, Z, 3, 3) diffusion tensor per voxel, already
                     scaled so white matter has higher/anisotropic diffusion
                     than gray matter (isotropic, lower magnitude)
        affine:      (4, 4) nibabel affine, needed to write outputs back out
                     in the same space as the source scan

    TODO: replace this stub with actual loading via nibabel + dipy:
        - load T1/FLAIR segmentation -> tissue_mask
        - load DTI -> fit tensor model (dipy.reconst.dti) -> D_tensor
        - typical scaling: D_white ~ 10x D_gray, both scaled by global D_scale
          at simulation time (see simulate_growth below)
    """
    raise NotImplementedError(
        "Wire this up to your actual UCSF-PDGM/UPenn-GBM loading pipeline. "
        "Placeholder below shows the expected shapes for testing."
    )


def make_synthetic_anatomy(shape=(64, 64, 64), seed=None):
    """
    Fallback placeholder anatomy generator, for testing the solver pipeline
    before real data is wired in. NOT used for actual paper results.
    """
    rng = np.random.default_rng(seed)
    tissue_mask = np.ones(shape, dtype=int)  # all gray matter by default

    # carve out a "white-matter"-like anisotropic band and a ventricle-like hole
    tissue_mask[shape[0] // 3: 2 * shape[0] // 3, :, :] = 2
    cx, cy, cz = shape[0] // 2, shape[1] // 4, shape[2] // 2
    zz, yy, xx = np.meshgrid(range(shape[2]), range(shape[1]), range(shape[0]), indexing="ij")
    ventricle = (xx - cx) ** 2 + (yy - cy) ** 2 + (zz - cz) ** 2 < 25
    tissue_mask[ventricle.transpose(2, 1, 0)] = 0

    D_tensor = np.zeros(shape + (3, 3))
    iso = np.eye(3) * 0.1
    aniso = np.diag([1.0, 0.1, 0.1])  # fast along x, mimicking a fiber direction
    for idx in np.ndindex(shape):
        D_tensor[idx] = aniso if tissue_mask[idx] == 2 else iso

    affine = np.eye(4)
    return tissue_mask, D_tensor, affine


# ---------------------------------------------------------------------------
# 2. Finite-volume Fisher-KPP solver (explicit time stepping)
# ---------------------------------------------------------------------------
def compute_divergence_diffusion(c, D_tensor, tissue_mask, voxel_size=1.0):
    """
    Compute div(D grad c) via a simple 6-neighbor finite-volume flux sum.
    Only the diagonal of D_tensor is used here for simplicity/speed; extend
    to full-tensor flux (using off-diagonal terms and neighbor-averaged D)
    if you need true fiber-crossing anisotropy in the synthetic data.
    """
    flux = np.zeros_like(c)
    for axis in range(3):
        D_axis = D_tensor[..., axis, axis]

        c_fwd = np.roll(c, -1, axis=axis)
        D_fwd = 0.5 * (D_axis + np.roll(D_axis, -1, axis=axis))
        flux += D_fwd * (c_fwd - c)

        c_bwd = np.roll(c, 1, axis=axis)
        D_bwd = 0.5 * (D_axis + np.roll(D_axis, 1, axis=axis))
        flux += D_bwd * (c_bwd - c)

    # zero-flux boundary: no growth into background (outside brain)
    flux[tissue_mask == 0] = 0.0
    return flux / (voxel_size ** 2)


def simulate_growth(tissue_mask, D_tensor, D_scale: float, rho: float,
                     seed_location, n_steps: int = 200, dt: float = 0.05,
                     save_every: int = 10):
    """
    Run the explicit Fisher-KPP simulation forward from a point seed.

    D_scale: global multiplier on the diffusion tensor field (randomize
             across synthetic patients, e.g. uniform in [0.05, 0.5] mm^2/day)
    rho:     proliferation rate (randomize, e.g. uniform in [0.005, 0.05] /day)
    seed_location: (x, y, z) index of initial tumor seed
    Returns: trajectory list of c arrays at intervals of `save_every` steps
    """
    c = np.zeros_like(tissue_mask, dtype=float)
    c[seed_location] = 1.0
    c = gaussian_filter(c, sigma=1.0)  # smooth initial seed into a small blob
    c[tissue_mask == 0] = 0.0

    D_scaled = D_tensor * D_scale
    trajectory = [c.copy()]

    for step in range(1, n_steps + 1):
        diffusion = compute_divergence_diffusion(c, D_scaled, tissue_mask)
        reaction = rho * c * (1 - c)
        c = c + dt * (diffusion + reaction)
        c = np.clip(c, 0.0, 1.0)
        c[tissue_mask == 0] = 0.0

        if step % save_every == 0:
            trajectory.append(c.copy())

    return trajectory


# ---------------------------------------------------------------------------
# 3. Batch generation driver
# ---------------------------------------------------------------------------
def generate_synthetic_dataset(output_dir: str, n_patients: int = 500,
                                use_real_anatomy: bool = False,
                                real_anatomy_dirs: list = None, seed: int = 0):
    """
    Generate n_patients synthetic growth trajectories.

    If use_real_anatomy=True, real_anatomy_dirs must be a list of patient
    directories (e.g. from UCSF-PDGM/UPenn-GBM) that load_patient_anatomy()
    can read; anatomies are sampled with replacement across n_patients so
    each gets a different randomized (D_scale, rho, seed_location) on
    possibly-repeated real anatomies -- this is intentional, since the
    variation you want in pretraining is growth-parameter variation, not
    anatomy variation (anatomy diversity comes from having many real subjects).
    """
    os.makedirs(output_dir, exist_ok=True)
    rng = np.random.default_rng(seed)

    for i in range(n_patients):
        if use_real_anatomy:
            anat_dir = rng.choice(real_anatomy_dirs)
            tissue_mask, D_tensor, affine = load_patient_anatomy(anat_dir)
        else:
            tissue_mask, D_tensor, affine = make_synthetic_anatomy(seed=int(rng.integers(1e6)))

        D_scale = rng.uniform(0.05, 0.5)
        rho = rng.uniform(0.005, 0.05)

        valid_voxels = np.argwhere(tissue_mask > 0)
        seed_idx = tuple(valid_voxels[rng.integers(len(valid_voxels))])

        trajectory = simulate_growth(tissue_mask, D_tensor, D_scale, rho, seed_idx)

        np.savez_compressed(
            os.path.join(output_dir, f"synthetic_patient_{i:04d}.npz"),
            trajectory=np.stack(trajectory),
            tissue_mask=tissue_mask,
            D_tensor=D_tensor,
            D_scale=D_scale,
            rho=rho,
            seed_location=seed_idx,
        )

        if (i + 1) % 50 == 0:
            print(f"Generated {i + 1}/{n_patients} synthetic trajectories")


# ---------------------------------------------------------------------------
# TODO checklist for your actual implementation:
# 1. load_patient_anatomy(): wire to real nibabel/dipy loading of
#    UCSF-PDGM / UPenn-GBM segmentations + DTI, replacing the raise above.
# 2. compute_divergence_diffusion(): currently uses only the diagonal of
#    D_tensor for speed; add full-tensor flux terms if fiber-crossing
#    anisotropy matters for your validation story.
# 3. Grid -> graph conversion: write a separate script that takes each
#    .npz here and converts the voxel grid into the supervoxel/parcel graph
#    format expected by pignn_glioma.py (x, edge_index, edge_attr, c0,
#    c_followup, boundary_mask). SLIC or a brain atlas parcellation are both
#    reasonable choices for the supervoxel step.
# 4. Consider replacing this finite-difference grid solver with a proper
#    FEM solver (FEniCS/SfePy) on a tetrahedral mesh if your GNN graph is
#    mesh-based rather than voxel-based -- keeps train/pretrain domains
#    consistent.
# ---------------------------------------------------------------------------


### 2b. `anatomy_and_graph_conversion.py`

In [ ]:
"""
Two pieces that bridge generate_synthetic_growth_data.py -> pignn_glioma.py:

1. load_patient_anatomy_real(): actual DTI loading + tensor fitting via
   nibabel/dipy, replacing the NotImplementedError stub.
2. grid_to_supervoxel_graph(): converts a voxel-grid trajectory + DTI tensor
   field into the torch_geometric Data object pignn_glioma.py expects
   (x, edge_index, edge_attr, c0, c_followup, boundary_mask), using SLIC
   supervoxels and the finite-volume conductance formula w_ij = (A_ij/h_ij^2)
   * n_ij^T D_ij n_ij from build_edge_features() in pignn_glioma.py.

Dependencies: numpy, scipy, scikit-image (SLIC), nibabel, dipy, torch,
torch_geometric. The graph-conversion half is runnable with just numpy/
scipy/scikit-image; the real DTI loader needs nibabel+dipy installed.
"""

import numpy as np
from scipy import ndimage
from skimage.segmentation import slic


# ---------------------------------------------------------------------------
# 1. Real DTI loading + tensor fitting (nibabel + dipy)
# ---------------------------------------------------------------------------
def load_patient_anatomy_real(dwi_path: str, bval_path: str, bvec_path: str,
                               seg_path: str, white_matter_scale: float = 10.0,
                               gray_matter_scale: float = 1.0):
    """
    Load diffusion-weighted MRI + b-values/vectors, fit a diffusion tensor
    model, and combine with a tissue segmentation to produce the
    (tissue_mask, D_tensor, affine) triple that generate_synthetic_growth_data.py
    and grid_to_supervoxel_graph() both expect.

    dwi_path:  4D DWI NIfTI (X, Y, Z, n_directions)
    bval_path: .bval file (b-values per direction)
    bvec_path: .bvec file (gradient directions)
    seg_path:  3D tissue segmentation NIfTI, integer labels matching your
               atlas convention (0=background/CSF, 1=gray matter,
               2=white matter, 3=ventricle -- adjust to your actual atlas)

    Returns: tissue_mask (X,Y,Z) int, D_tensor (X,Y,Z,3,3) float, affine (4,4)

    NOTE: requires `pip install nibabel dipy --break-system-packages`.
    Not runnable in this environment (no network access to install), but
    syntax-verified; this is the real implementation to drop in once you
    have DTI data + these packages available.
    """
    import nibabel as nib
    from dipy.core.gradients import gradient_table
    from dipy.reconst.dti import TensorModel

    dwi_img = nib.load(dwi_path)
    dwi_data = dwi_img.get_fdata()
    affine = dwi_img.affine

    gtab = gradient_table(bval_path, bvec_path)
    tensor_model = TensorModel(gtab)
    tensor_fit = tensor_model.fit(dwi_data)

    # dipy's quadratic_form gives the full symmetric 3x3 tensor per voxel
    D_tensor_raw = tensor_fit.quadratic_form  # (X, Y, Z, 3, 3), physical units

    seg_img = nib.load(seg_path)
    tissue_mask = seg_img.get_fdata().astype(int)

    # Rescale the fitted tensor by tissue type so white matter shows the
    # expected faster/more anisotropic invasion pathway; fitted DTI values
    # are noisy at the individual-voxel level, so blending with a
    # tissue-type prior like this is standard practice in this literature
    # rather than trusting raw per-voxel tensors alone.
    D_tensor = D_tensor_raw.copy()
    D_tensor[tissue_mask == 2] *= white_matter_scale
    D_tensor[tissue_mask == 1] *= gray_matter_scale
    D_tensor[tissue_mask == 0] = 0.0  # no diffusion outside brain/in CSF

    return tissue_mask, D_tensor, affine


# ---------------------------------------------------------------------------
# 2. Voxel grid -> supervoxel graph conversion
# ---------------------------------------------------------------------------
def compute_supervoxels(tissue_mask, c0, n_segments: int = 500, compactness: float = 0.1):
    """
    Run SLIC supervoxel segmentation restricted to brain tissue.
    Using c0 (baseline tumor density) as part of the SLIC input image
    biases supervoxel boundaries to respect the tumor edge, which matters
    since you don't want a supervoxel straddling healthy/tumor tissue.
    """
    brain_mask = tissue_mask > 0
    # Stack tissue label + baseline density as a 2-channel "image" for SLIC
    slic_input = np.stack([tissue_mask.astype(float), c0], axis=-1)
    labels = slic(slic_input, n_segments=n_segments, compactness=compactness,
                  mask=brain_mask, channel_axis=-1, start_label=0)
    return labels  # (X, Y, Z) int, -1 outside mask (skimage uses 0 as background if mask given)


def build_supervoxel_graph(labels, tissue_mask, D_tensor, c0, c_followup,
                            voxel_size=1.0):
    """
    Aggregate voxel-grid quantities into per-supervoxel node features and
    compute edge weights via the finite-volume conductance formula
    w_ij = (A_ij / h_ij^2) * n_ij^T D_ij n_ij, matching build_edge_features()
    in pignn_glioma.py so the two pipelines are numerically consistent.

    voxel_size: either a scalar (isotropic pixel-unit spacing -- the old,
    still-supported default, since RHUH-GBM's 2D slices come from PNG-style
    images with no NIfTI affine to pull real spacing from) or a length-3
    array/sequence (dx, dy, dz) of REAL physical spacing in mm, typically
    `nib.load(path).header.get_zooms()[:3]` for a NIfTI volume. This matters
    because the conductance formula divides by h_ij^2, the physical distance
    between supervoxel centroids -- if h_ij is computed in raw pixel-index
    units instead of mm (the previous behavior for every real patient, not
    just RHUH-GBM), w_ij is off by whatever factor separates pixel spacing
    from true spacing, for every edge, in every graph. `voxel_size` is
    broadcast against a (3,) centroid coordinate below, so both a scalar and
    a length-3 array work with no other change needed.

    Returns a plain-dict graph (convert to torch_geometric.data.Data at the
    call site, once torch is available):
        node_centroids: (N, 3)
        node_features:  (N, F)  [mean tissue label, mean c0, is-boundary flag]
        c0_agg:         (N,)    mean baseline density per supervoxel
        c_followup_agg: (N,)    mean follow-up density per supervoxel
        edge_index:     (2, E)
        edge_attr:      (E, 2)  [w_ij, h_ij]  (column 0 = physics weight, matches
                                 the convention expected by ReactionDiffusionStep)
        boundary_mask:  (N,) bool, True for supervoxels touching the brain edge
    """
    voxel_size = np.asarray(voxel_size, dtype=float)  # scalar or (3,) both broadcast correctly below

    unique_labels = np.unique(labels[labels >= 0])
    n_nodes = len(unique_labels)
    label_to_idx = {lab: i for i, lab in enumerate(unique_labels)}

    node_centroids = np.zeros((n_nodes, 3))
    node_features = np.zeros((n_nodes, 3))
    raw_sizes = np.zeros(n_nodes)
    c0_agg = np.zeros(n_nodes)
    c_followup_agg = np.zeros(n_nodes)
    boundary_mask = np.zeros(n_nodes, dtype=bool)
    node_D = np.zeros((n_nodes, 3, 3))

    brain_eroded = ndimage.binary_erosion(tissue_mask > 0)
    boundary_voxels = (tissue_mask > 0) & (~brain_eroded)

    for lab in unique_labels:
        idx = label_to_idx[lab]
        voxel_mask = labels == lab
        coords = np.argwhere(voxel_mask)
        node_centroids[idx] = coords.mean(axis=0) * voxel_size
        raw_sizes[idx] = voxel_mask.sum()
        node_features[idx, 0] = tissue_mask[voxel_mask].mean()
        node_features[idx, 1] = c0[voxel_mask].mean()
        c0_agg[idx] = c0[voxel_mask].mean()
        c_followup_agg[idx] = c_followup[voxel_mask].mean()
        boundary_mask[idx] = boundary_voxels[voxel_mask].any()
        node_D[idx] = D_tensor[voxel_mask].mean(axis=0)

    # Relative supervoxel size (size / mean size across this graph), not the raw
    # voxel count. The raw count is O(10-1000) depending on image resolution and
    # n_segments, while the other two features are O(1) -- at a Linear layer's
    # default init (calibrated for unit-scale inputs) that mismatch alone produces
    # huge pre-activation values, causing exactly the exploding-loss / dead-network
    # behavior (sigmoid/softplus saturation, frozen gradients) observed when this
    # graph was actually trained on, rather than just forward-passed once. Relative
    # size is also more meaningful than a raw count: "1.5x the median supervoxel"
    # is informative and scale-invariant across patients/resolutions, "230 voxels"
    # is neither.
    mean_size = raw_sizes.mean() if raw_sizes.mean() > 0 else 1.0
    node_features[:, 2] = raw_sizes / mean_size

    # Build edges between spatially adjacent supervoxels (shared face in the
    # voxel grid = adjacency in the graph)
    edges = set()
    shifts = [(1, 0, 0), (0, 1, 0), (0, 0, 1)]
    for dx, dy, dz in shifts:
        shifted = np.roll(labels, shift=(-dx, -dy, -dz), axis=(0, 1, 2))
        valid = (labels >= 0) & (shifted >= 0) & (labels != shifted)
        pairs = np.stack([labels[valid], shifted[valid]], axis=-1)
        for a, b in np.unique(pairs, axis=0):
            if a in label_to_idx and b in label_to_idx:
                edges.add((label_to_idx[a], label_to_idx[b]))
                edges.add((label_to_idx[b], label_to_idx[a]))  # undirected -> both directions

    edge_index = np.array(list(edges)).T  # (2, E)

    # Finite-volume conductance per edge, same formula as pignn_glioma.py's
    # build_edge_features(), but computed here from the aggregated node_D
    src, dst = edge_index
    diff = node_centroids[dst] - node_centroids[src]
    h_ij = np.linalg.norm(diff, axis=-1) + 1e-8
    n_ij = diff / h_ij[:, None]

    D_mid = 0.5 * (node_D[src] + node_D[dst])
    Dn = np.einsum("eij,ej->ei", D_mid, n_ij)
    n_D_n = np.einsum("ei,ei->e", n_ij, Dn)

    # NOTE: shared boundary area A_ij is approximated here as a constant
    # (real implementation should count actual shared-face voxel area between
    # the two supervoxels, similar to the edge-building loop above -- left
    # as a refinement since it requires tracking per-pair face counts)
    A_ij_approx = 1.0
    w_ij = (A_ij_approx / h_ij ** 2) * n_D_n

    edge_attr = np.stack([w_ij, h_ij], axis=-1)

    return {
        "node_centroids": node_centroids,
        "node_features": node_features,
        "c0": c0_agg,
        "c_followup": c_followup_agg,
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "boundary_mask": boundary_mask,
    }


# ---------------------------------------------------------------------------
# Smoke test with synthetic grid data (no torch/dipy required)
# ---------------------------------------------------------------------------


### 2c. `pignn_glioma.py`

In [ ]:
"""
Physics-Informed Graph Neural Network (PI-GNN) for glioma growth prediction.

Core idea
---------
Each graph node = a supervoxel / atlas parcel of brain tissue, carrying a
scalar tumor cell density c_i in [0, 1]. Edge weights are precomputed from
DTI (anisotropic diffusion tensor), NOT learned from scratch -- they encode
the finite-volume discretization of div(D grad c). A small MLP learns a
patient/tissue-specific *correction* to those physics-derived weights and to
the per-node proliferation rate rho_i. Message passing = one explicit time
step of the Fisher-KPP reaction-diffusion PDE:

    dc/dt = div(D grad c) + rho * c * (1 - c)

This file is a SKELETON: data loading, DTI edge-weight precomputation, and
FEM synthetic-data generation are stubbed out with clear TODOs. Fill those
in with your actual preprocessing pipeline (nibabel / dipy for DTI, your
FEM solver for synthetic trajectories).

Dependencies: torch, torch_geometric
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


# ---------------------------------------------------------------------------
# 1. Physics-informed message passing layer
# ---------------------------------------------------------------------------
class ReactionDiffusionStep(MessagePassing):
    """
    One explicit time step of:
        c_i(t+1) = c_i(t) + dt * [ sum_j w_ij_learned * (c_j - c_i)
                                    + rho_i_learned * c_i * (1 - c_i) ]

    w_ij (base) is the physics-derived diffusion conductance from DTI.
    w_ij_learned = w_ij * sigmoid(MLP_theta(edge_features))
    rho_i_learned = softplus(MLP_phi(node_features))
    """

    def __init__(self, node_feat_dim: int, edge_feat_dim: int, hidden_dim: int = 32):
        # node_dim=0 (not the MessagePassing default of -2): c is a plain 1D
        # (N,) density tensor, not the (N, F) feature convention the default
        # assumes, so -2 is out of range on c. node_dim=0 also correctly
        # covers edge_attr's (E, F) axis, so it works for both.
        super().__init__(aggr="add", node_dim=0)  # sum over neighbors, matches PDE discretization

        # Edge correction MLP: takes [edge_features, c_i, c_j] -> scalar gate in (0,1)
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_feat_dim + 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

        # Node proliferation-rate MLP: takes static node features -> rho_i >= 0
        self.rho_mlp = nn.Sequential(
            nn.Linear(node_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

        self.dt = 0.1  # discrete time step; treat as hyperparameter / could be learned
        self._last_gate = None  # stashed by message() each forward call; read by physics_prior_loss below

    def forward(self, c, node_features, edge_index, edge_attr, rho_scale=1.0):
        """
        c:              (N,) current tumor density per node
        node_features:  (N, node_feat_dim) static features (tissue type, distance to
                         resection cavity, baseline segmentation, etc.)
        edge_index:     (2, E) graph connectivity
        edge_attr:      (E, edge_feat_dim) includes the physics-derived base
                         conductance w_ij as one of the features -- see
                         build_edge_features() below
        rho_scale:      scalar (float or 0-dim/1-dim tensor, e.g. an
                         nn.Parameter being optimized by inverse_fit_patient())
                         multiplying the learned proliferation rate. Default
                         1.0 reproduces the previous behavior exactly for
                         every existing call site (train_multi_patient(),
                         evaluate_model_on_patients(), etc.) that doesn't
                         pass it.
        """
        diffusion_update = self.propagate(edge_index, c=c, edge_attr=edge_attr)

        rho = F.softplus(self.rho_mlp(node_features)).squeeze(-1) * rho_scale  # (N,)
        reaction_update = rho * c * (1 - c)

        c_next = c + self.dt * (diffusion_update + reaction_update)
        return c_next.clamp(0.0, 1.0), rho  # clamp keeps density physically valid

    def message(self, c_i, c_j, edge_attr):
        # edge_attr[:, 0] is assumed to be the precomputed physics base weight w_ij
        w_base = edge_attr[:, 0]
        gate = torch.sigmoid(
            self.edge_mlp(torch.cat([edge_attr, c_i.unsqueeze(-1), c_j.unsqueeze(-1)], dim=-1))
        ).squeeze(-1)
        self._last_gate = gate  # exposed for physics_prior_loss -- see note there
        w_learned = w_base * gate
        return w_learned * (c_j - c_i)


# ---------------------------------------------------------------------------
# 2. Full model: unroll the PDE step over T timesteps
# ---------------------------------------------------------------------------
class GliomaGrowthPIGNN(nn.Module):
    def __init__(self, node_feat_dim: int, edge_feat_dim: int, hidden_dim: int = 32):
        super().__init__()
        self.step = ReactionDiffusionStep(node_feat_dim, edge_feat_dim, hidden_dim)

    def forward(self, c0, node_features, edge_index, edge_attr, n_steps: int, rho_scale=1.0):
        """
        Simulate forward from baseline density c0 for n_steps.
        Returns the full trajectory (for the PDE residual loss) and the
        final state (for the data-fit loss against the follow-up scan).

        rho_scale: passed straight through to each ReactionDiffusionStep
        call; see its docstring. Default 1.0 keeps every existing call site
        unchanged.
        """
        c = c0
        trajectory = [c]
        rhos = []
        for _ in range(n_steps):
            c, rho = self.step(c, node_features, edge_index, edge_attr, rho_scale=rho_scale)
            trajectory.append(c)
            rhos.append(rho)
        return torch.stack(trajectory, dim=0), torch.stack(rhos, dim=0)  # (T+1, N), (T, N)


# ---------------------------------------------------------------------------
# 3. Losses
# ---------------------------------------------------------------------------
def dice_loss(pred, target, eps: float = 1e-6):
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + eps) / (pred.sum() + target.sum() + eps)


def physics_prior_loss(trajectory, node_features, edge_index, edge_attr, step_module, dt):
    """
    Physics-prior regularizer on the LEARNED corrections (replaces an earlier
    "pde_residual_loss" that was mathematically vacuous -- see note below for
    why, kept here so the failure mode isn't silently lost to history).

    Why the original formulation was a no-op:
    GliomaGrowthPIGNN.forward() generates `trajectory` by unrolling
    step_module itself: c_next = c + dt * (diffusion(c) + reaction(c)), and
    trajectory[t+1] IS c_next. Recomputing step_module(trajectory[t]) and
    comparing the result to trajectory[t+1] therefore just repeats the exact
    arithmetic that produced trajectory[t+1] in the first place -- the
    "residual" is 0 to floating-point precision for every timestep and every
    set of weights, trained or not (confirmed empirically on real RHUH-GBM
    and LUMIERE graphs: pde=0.0 exactly). It contributed no gradient signal.
    A genuine PDE-residual term only makes sense when the trajectory being
    checked is NOT itself generated by literally executing the same operator
    being checked against -- e.g. free/latent intermediate states in a
    collocation-style PINN. This architecture instead mechanistically
    simulates the trajectory, so that formulation doesn't apply here.

    What this regularizes instead:
    the two corrections ReactionDiffusionStep learns on top of the
    DTI-derived physics -- the per-edge diffusion gate
    sigmoid(edge_mlp(...)) in (0, 1) (1 = "trust the physics-derived
    conductance as-is"), and the per-node proliferation rate rho from
    rho_mlp (0 = "no learned growth beyond diffusion"). Penalizing
    (gate - 1)^2 and rho^2 anchors the simulation to the DTI physics prior
    unless L_data (the fit to the real follow-up scan) pulls it away --
    which is the actual behavior wanted here: physically-plausible dynamics
    between the two sparse real observations, not a literal PDE residual.
    """
    gate_penalties = []
    rho_penalties = []
    for t in range(trajectory.shape[0] - 1):
        c_t = trajectory[t]
        _, rho = step_module(c_t, node_features, edge_index, edge_attr)
        gate = step_module._last_gate  # set as a side effect of message() inside the call above
        gate_penalties.append(F.mse_loss(gate, torch.ones_like(gate)))
        rho_penalties.append((rho ** 2).mean())
    return torch.stack(gate_penalties).mean() + torch.stack(rho_penalties).mean()


def boundary_loss(c_final, boundary_mask):
    """Zero-flux / zero-density boundary at skull and ventricle nodes."""
    return (c_final[boundary_mask] ** 2).mean()


def total_loss(trajectory, c_observed_final, node_features, edge_index, edge_attr,
                step_module, boundary_mask, lambdas=(1.0, 0.5, 0.1)):
    lam_data, lam_prior, lam_bc = lambdas
    c_final = trajectory[-1]

    L_data = dice_loss(c_final, c_observed_final)
    L_prior = physics_prior_loss(trajectory, node_features, edge_index, edge_attr,
                                  step_module, dt=step_module.dt)
    L_bc = boundary_loss(c_final, boundary_mask)

    return lam_data * L_data + lam_prior * L_prior + lam_bc * L_bc, {
        "data": L_data.item(), "physics_prior": L_prior.item(), "bc": L_bc.item()
    }


# ---------------------------------------------------------------------------
# 4. Edge feature construction (physics precomputation) -- TODO: fill in
# ---------------------------------------------------------------------------
def build_edge_features(node_centroids, diffusion_tensors, edge_index, shared_boundary_area):
    """
    Compute the finite-volume conductance w_ij = (A_ij / h_ij^2) * n_ij^T D_ij n_ij
    for every edge, from DTI-derived diffusion tensors.

    node_centroids:     (N, 3) node positions in physical space (mm)
    diffusion_tensors:  (N, 3, 3) DTI tensor per node (interpolate to edge midpoint)
    edge_index:         (2, E)
    shared_boundary_area: (E,) precomputed shared face area between adjacent supervoxels

    Returns: edge_attr (E, edge_feat_dim), where column 0 = w_ij (base physics weight)
    """
    src, dst = edge_index
    diff = node_centroids[dst] - node_centroids[src]
    h_ij = diff.norm(dim=-1) + 1e-8
    n_ij = diff / h_ij.unsqueeze(-1)

    D_mid = 0.5 * (diffusion_tensors[src] + diffusion_tensors[dst])  # (E, 3, 3)
    Dn = torch.einsum("eij,ej->ei", D_mid, n_ij)
    n_D_n = torch.einsum("ei,ei->e", n_ij, Dn)

    w_ij = (shared_boundary_area / h_ij.pow(2)) * n_D_n

    # TODO: append additional edge features here (e.g. tissue-type mismatch
    # flag, distance itself) -- keep w_ij as column 0 since message() expects it
    edge_attr = torch.stack([w_ij, h_ij], dim=-1)
    return edge_attr


# ---------------------------------------------------------------------------
# 5. Training loop skeleton
# ---------------------------------------------------------------------------
def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        trajectory, _ = model(
            c0=batch.c0,
            node_features=batch.x,
            edge_index=batch.edge_index,
            edge_attr=batch.edge_attr,
            n_steps=batch.n_steps[0].item(),  # assumes uniform n_steps per batch
        )

        loss, components = total_loss(
            trajectory, batch.c_followup, batch.x, batch.edge_index, batch.edge_attr,
            model.step, batch.boundary_mask,
        )
        loss.backward()
        optimizer.step()
        total += loss.item()

    return total / len(loader)


def inverse_fit_patient(model, x, edge_index, edge_attr, c0, c_followup, boundary_mask,
                         n_steps: int, n_iters: int = 200, lr: float = 1e-2):
    """
    Freeze theta/phi (already trained). Fit a single per-patient scalar
    (rho_scale, multiplying the learned proliferation rate) by backprop
    against that patient's own baseline + follow-up scan. This replaces slow
    iterative FEM fitting with a single fast gradient-based fit.

    Previously a placeholder: rho_scale was optimized against a loss that
    never actually depended on it (model(...) was called without rho_scale,
    so every gradient step no-opped and the routine could not produce a
    usable per-patient fit). Fixed now that ReactionDiffusionStep.forward()
    and GliomaGrowthPIGNN.forward() both accept and apply rho_scale (see
    their docstrings) -- rho_scale is passed through on every call below, so
    the loss genuinely depends on it and the Adam step has a real gradient
    to follow.

    Takes the same (x, edge_index, edge_attr, c0, c_followup, boundary_mask)
    tuple that graph_dict_to_tensors() returns elsewhere in this pipeline,
    rather than a torch_geometric Data object with those as attributes --
    matches how every other function in this file/notebook is actually
    called on real patient graphs.
    """
    for p in model.parameters():
        p.requires_grad_(False)

    rho_scale = torch.nn.Parameter(torch.tensor(1.0, device=c0.device))
    optimizer = torch.optim.Adam([rho_scale], lr=lr)

    for _ in range(n_iters):
        optimizer.zero_grad()
        trajectory, _ = model(c0, x, edge_index, edge_attr, n_steps=n_steps, rho_scale=rho_scale)
        loss, _ = total_loss(trajectory, c_followup, x, edge_index, edge_attr,
                              model.step, boundary_mask)
        loss.backward()
        optimizer.step()

    return rho_scale.item()


# ---------------------------------------------------------------------------
# TODO checklist for your actual implementation:
# 1. build_edge_features(): load DTI via dipy/nibabel, compute per-supervoxel
#    mean tensor, wire in real shared_boundary_area from your parcellation.
# 2. Dataset class: wrap RHUH-GBM / LUMIERE / UCSF-PDGM into torch_geometric
#    Data objects with fields: x, edge_index, edge_attr, c0, c_followup,
#    boundary_mask, n_steps.
# 3. FEM synthetic data generator: separate script, not shown here, to
#    pretrain before fine-tuning on real longitudinal scans.
# (inverse_fit_patient()'s rho_scale threading is fixed above, and now
#  exercised in the Section 8 CV loop via evaluate_personalized_pi(): each
#  fold's frozen model_pi gets a per-patient rho_scale fit on its own
#  held-out validation patients, compared against zero-shot PI-GNN and a
#  per-patient FEM fit on the same patients.)
# ---------------------------------------------------------------------------


## 3. Dataset discovery

Prints what's actually on disk for the attached Kaggle dataset -- adjust the `find_*` glob patterns in Section 6 below if your mirror's layout differs from what's printed.


In [ ]:
def list_dataset_tree(root, max_files=25):
    if not os.path.isdir(root):
        print(f"  (not found: {root})")
        return
    count = 0
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > 3:
            continue
        for fn in sorted(filenames):
            print(os.path.relpath(os.path.join(dirpath, fn), root))
            count += 1
            if count >= max_files:
                print(f"... (truncated after {max_files} files; increase max_files to see more)")
                return

def find_dataset_root(slug):
    """
    Kaggle has mounted attached datasets at two different path conventions
    depending on notebook/UI version: the classic /kaggle/input/<slug>/, and a
    newer nested /kaggle/input/datasets/<owner>/<slug>/ (confirmed on a real run --
    the classic path did not exist, only the nested one did, under owner
    "rodschermitg" for LUMIERE). Try both rather than hard-coding either one, so
    this keeps working if Kaggle's convention or the owner changes.
    """
    candidates = [os.path.join(KAGGLE_INPUT, slug)]
    candidates += sorted(glob.glob(os.path.join(KAGGLE_INPUT, "datasets", "*", slug)))
    for c in candidates:
        if os.path.isdir(c):
            return c
    print(f"WARNING: could not find dataset \'{slug}\' under any of: {candidates}")
    print(f"  Run `ls {KAGGLE_INPUT}` and `ls {KAGGLE_INPUT}/datasets/*` in a scratch cell "
          f"to see what's actually attached, then adjust find_dataset_root() above.")
    return candidates[0]  # fall back to the classic path so downstream code still gets a string

LUMIERE_ROOT = find_dataset_root("lumiere-dataset")

print("=" * 70)
print("LUMIERE root:", LUMIERE_ROOT)
print("=" * 70)
list_dataset_tree(LUMIERE_ROOT)

# Generic, defensive filename search used by the LUMIERE loader below --
# real dataset mirrors are reorganized between versions, so this stays
# broad (substring match) rather than hard-coded to one exact layout.
def find_first(root, must_contain_all=(), exts=(".nii", ".nii.gz"), exclude=()):
    must_contain_all = [s.lower() for s in must_contain_all]
    exclude = [s.lower() for s in exclude]
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            low = fn.lower()
            if not low.endswith(exts):
                continue
            if any(tok in low for tok in exclude):
                continue
            if all(tok in low for tok in must_contain_all):
                return os.path.join(dirpath, fn)
    return None


## 4. Test A — RHUH-GBM (real segmentation → 2D supervoxel graph)

Confirmed structure (checked directly on the Hugging Face dataset viewer): 120 rows, columns `patient_id`, `timepoint`, `timepoint_name` (`preop` / `early_postop` / `followup_recurrence`), `labels_present`, and JPEG-encoded `t1ce`, `flair`, `segmentation`, `overlay` images (a single representative 2D slice per row, not a full 3D volume). Good for a fast, real-segmentation sanity check of `compute_supervoxels()` / `build_supervoxel_graph()` in 2D; not suited to the 3D DTI-driven solver (no DWI here, and only one slice per scan).

In [ ]:
from datasets import load_dataset
from PIL import Image
import collections

rhuh = load_dataset("MedOtter/RHUH-GBM", split="train")
print(rhuh)
print(rhuh[0].keys())

by_patient = collections.defaultdict(dict)
for row in rhuh:
    by_patient[row["patient_id"]][row["timepoint_name"]] = row

# Need a baseline -> later-timepoint pair per patient. Prefer preop -> the actual
# recurrence follow-up; fall back to preop -> early_postop (still a real, if less
# clinically interesting, c0 -> c_followup transition) rather than dropping the
# patient outright, so the training cohort isn't smaller than it needs to be.
FOLLOWUP_PRIORITY = ["followup_recurrence", "early_postop"]

patient_pairs = {}
for pid, tps in by_patient.items():
    if "preop" not in tps:
        continue
    for fu_name in FOLLOWUP_PRIORITY:
        if fu_name in tps:
            patient_pairs[pid] = (tps["preop"], tps[fu_name], fu_name)
            break

print(f"\nTotal distinct patients: {len(by_patient)}")
print(f"Usable preop -> followup pairs: {len(patient_pairs)}")
for fu_name in FOLLOWUP_PRIORITY:
    n = sum(1 for _, _, f in patient_pairs.values() if f == fu_name)
    print(f"  via preop -> {fu_name}: {n}")


In [ ]:
def seg_and_density_from_row(row):
    seg = np.array(row["segmentation"].convert("L"))
    # BraTS-style label convention: >0 = some tumor sub-region: treat any nonzero as tumor density 1.0
    density = (seg > 0).astype(float)
    tissue_mask = np.ones_like(seg, dtype=int)  # 2D slice: brain tissue = 1 everywhere in-frame (no CSF/WM split here)
    return tissue_mask, density

def build_rhuh_graph(row_pre, row_fu, n_segments=150):
    tissue_2d, c0_2d = seg_and_density_from_row(row_pre)
    _, c_followup_2d = seg_and_density_from_row(row_fu)
    if c0_2d.shape != c_followup_2d.shape:
        return None, f"shape mismatch {c0_2d.shape} vs {c_followup_2d.shape}"
    if c0_2d.sum() < 5 or c_followup_2d.sum() < 5:
        return None, "baseline or follow-up mask has under 5 nonzero pixels"

    # grid_to_supervoxel_graph's helpers are written for 3D (X,Y,Z) volumes; add a
    # singleton depth axis so the exact same 3D code path runs unmodified on this 2D slice.
    tissue_3d, c0_3d, c_followup_3d = tissue_2d[:, :, None], c0_2d[:, :, None], c_followup_2d[:, :, None]
    D_tensor_3d = np.zeros(tissue_3d.shape + (3, 3))
    D_tensor_3d[tissue_3d > 0] = np.eye(3) * 0.1  # no DTI on a 2D slice mirror -- isotropic placeholder

    labels_2d = compute_supervoxels(tissue_3d, c0_3d, n_segments=n_segments)
    # voxel_size intentionally left at build_supervoxel_graph's default (pixel-unit,
    # not real mm): this HuggingFace image dataset's rows carry a "shape" field but
    # no pixel-spacing/affine, so there is no real spacing to pull here the way
    # build_lumiere_graph() now does from the NIfTI header. RHUH-GBM's physics
    # units remain approximate for this reason -- see Limitations.
    graph = build_supervoxel_graph(labels_2d, tissue_3d, D_tensor_3d, c0_3d, c_followup_3d)
    if graph["node_centroids"].shape[0] < 5:
        return None, "too few supervoxels after SLIC (degenerate mask?)"
    return graph, "ok"

rhuh_graphs = {}
rhuh_skipped = []
for pid, (row_pre, row_fu, fu_name) in patient_pairs.items():
    try:
        graph, note = build_rhuh_graph(row_pre, row_fu)
    except Exception as e:
        graph, note = None, f"exception: {e}"
    if graph is not None:
        rhuh_graphs[pid] = graph
    else:
        rhuh_skipped.append((pid, note))

print(f"Built {len(rhuh_graphs)} / {len(patient_pairs)} usable RHUH-GBM patient graphs.")
if rhuh_skipped:
    print("Skipped patients (reason):")
    for pid, note in rhuh_skipped[:15]:
        print(f"  {pid}: {note}")
    if len(rhuh_skipped) > 15:
        print(f"  ... and {len(rhuh_skipped) - 15} more")


In [ ]:
RHUH_PATIENT_IDS = sorted(rhuh_graphs.keys())
N_RHUH = len(RHUH_PATIENT_IDS)
N_FOLDS = min(5, N_RHUH) if N_RHUH >= 2 else 1

# Stratify roughly by baseline tumor size (sum of c0 across supervoxels) so no
# fold is dominated by unusually large/small tumors, per the paper's 4.1 design --
# approximated as sort-then-round-robin-assign rather than a full stratified
# k-fold, to avoid depending on sklearn's binning for what may be a very small n.
sizes = {pid: float(rhuh_graphs[pid]["c0"].sum()) for pid in RHUH_PATIENT_IDS}
sorted_pids = sorted(RHUH_PATIENT_IDS, key=lambda p: sizes[p])
rhuh_folds = [sorted_pids[i::N_FOLDS] for i in range(N_FOLDS)]

print(f"{N_RHUH} usable patients -> {N_FOLDS} folds:")
for i, f in enumerate(rhuh_folds):
    print(f"  fold {i}: {len(f)} patients")

if RHUH_PATIENT_IDS:
    demo_pid = RHUH_PATIENT_IDS[0]
    demo_graph = rhuh_graphs[demo_pid]
    row_pre_demo, row_fu_demo, _ = patient_pairs[demo_pid]
    tissue_2d_demo, c0_2d_demo = seg_and_density_from_row(row_pre_demo)
    labels_2d_demo = compute_supervoxels(tissue_2d_demo[:, :, None], c0_2d_demo[:, :, None], n_segments=150)

    print(f"\nDemo patient: {demo_pid}  supervoxels={demo_graph['node_centroids'].shape[0]}  "
          f"edges={demo_graph['edge_index'].shape[1]}")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(np.array(row_pre_demo["t1ce"].convert("L")), cmap="gray"); axes[0].set_title("T1ce (pre-op)")
    axes[1].imshow(c0_2d_demo, cmap="hot"); axes[1].set_title("c0 (pre-op tumor mask)")
    axes[2].imshow(labels_2d_demo[:, :, 0], cmap="tab20"); axes[2].set_title(f"{demo_graph['node_centroids'].shape[0]} supervoxels")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

# kept for compatibility with the Section 6 forward-pass smoke test below, which
# expects a single demo graph named graph_rhuh
if rhuh_graphs:
    graph_rhuh = rhuh_graphs[RHUH_PATIENT_IDS[0]]


## 5. Test B — LUMIERE (real longitudinal baseline → follow-up pair)

LUMIERE has real pre-/post-contrast T1, T2, FLAIR and automated segmentations across multiple real follow-up timepoints per patient -- no DTI needed here, since this is testing the "real `c0` -> real `c_followup`" path directly (the external-validation role Table 1 assigns it in the paper), not the synthetic-pretraining path.


In [ ]:
def find_lumiere_patient_dirs(root):
    patient_dirs = []
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if "patient" in d.lower():
                patient_dirs.append(os.path.join(dirpath, d))
    return sorted(set(patient_dirs))

lumiere_patients = find_lumiere_patient_dirs(LUMIERE_ROOT)
print(f"Found {len(lumiere_patients)} candidate patient directories under LUMIERE.")
if lumiere_patients:
    print("Example:", lumiere_patients[0])

def find_lumiere_timepoint_dirs(patient_dir):
    # LUMIERE timepoints are typically dated/weekly subfolders under each patient dir.
    return sorted([os.path.join(patient_dir, d) for d in os.listdir(patient_dir)
                   if os.path.isdir(os.path.join(patient_dir, d))])


In [ ]:
def find_in_subfolder(root, folder_token, exts=(".nii", ".nii.gz")):
    # Looks for a NIfTI file inside any subfolder whose NAME contains folder_token
    # -- used to target LUMIERE's named segmentation-method output folders
    # rather than guessing from the filename alone.
    for dirpath, _, filenames in os.walk(root):
        if folder_token in os.path.basename(dirpath).lower():
            for fn in sorted(filenames):
                if fn.lower().endswith(exts):
                    return os.path.join(dirpath, fn)
    return None

def find_lumiere_seg(tp_dir):
    # LUMIERE segmentations come from two named pipelines (DeepBraTumIA,
    # HD-GLIO-AUTO) -- prefer their dedicated output folders first. A bare
    # "seg" filename match is a last resort and explicitly excludes
    # "tissue_seg", which is a whole-brain CSF/GM/WM tissue segmentation,
    # NOT a tumor mask (confirmed on a real LUMIERE patient: labels 1/2/3
    # covering ~14% of the volume -- far too large to be a glioma lesion).
    return (find_in_subfolder(tp_dir, "deepbratumia")
            or find_in_subfolder(tp_dir, "hd-glio")
            or find_first(tp_dir, ["tumor", "seg"])
            or find_first(tp_dir, ["lesion", "seg"])
            or find_first(tp_dir, ["seg"], exclude=["tissue"]))

def build_lumiere_graph(patient_dir, verbose=False):
    """
    Returns (graph_dict, note) for one LUMIERE patient, or (None, reason) if
    this patient cannot be used. note/reason are short diagnostic strings
    printed by the loop below.
    """
    timepoints = find_lumiere_timepoint_dirs(patient_dir)
    if len(timepoints) < 2:
        return None, f"only {len(timepoints)} timepoint(s)"

    seg_baseline = find_lumiere_seg(timepoints[0])
    seg_followup = find_lumiere_seg(timepoints[-1])
    if not (seg_baseline and seg_followup):
        return None, "no segmentation found at baseline and/or follow-up"

    import nibabel as nib
    seg_b_img = nib.load(seg_baseline)
    seg_b = seg_b_img.get_fdata()
    seg_f = nib.load(seg_followup).get_fdata()
    # Real per-axis voxel spacing (mm) from the NIfTI header, not the pixel-unit
    # default used everywhere else in this pipeline -- the finite-volume
    # conductance formula divides by h_ij^2 (physical centroid distance), so
    # using pixel-index spacing instead of real mm makes every edge weight
    # wrong by whatever factor separates the two. RHUH-GBM's 2D slices come
    # from a HuggingFace image dataset with no NIfTI affine, so this fix only
    # applies to LUMIERE for now (see build_rhuh_graph()/Limitations).
    voxel_spacing_mm = np.array(seg_b_img.header.get_zooms()[:3], dtype=float)

    if verbose:
        vals_b, counts_b = np.unique(seg_b, return_counts=True)
        print(f"  baseline seg file: {seg_baseline}")
        print(f"  baseline seg label histogram: {dict(zip(vals_b.tolist(), counts_b.tolist()))}")
        frac_b = (seg_b > 0).sum() / seg_b.size
        print(f"  baseline nonzero-label fraction of volume: {frac_b:.3f}")
        if frac_b > 0.10:
            print("  WARNING: >10% of volume nonzero -- looks like a brain/tissue mask, not a tumor mask.")

    if seg_b.shape != seg_f.shape:
        return None, f"baseline/follow-up shape mismatch {seg_b.shape} vs {seg_f.shape} (needs registration)"

    c0 = (seg_b > 0).astype(float)
    c_followup = (seg_f > 0).astype(float)
    if c0.sum() < 5 or c_followup.sum() < 5:
        return None, "baseline or follow-up tumor mask has under 5 voxels"

    tissue = np.ones_like(c0, dtype=int)
    D = np.zeros(tissue.shape + (3, 3))
    D[tissue > 0] = np.eye(3) * 0.1  # isotropic placeholder -- LUMIERE has no DTI, see Section 0 notes

    labels = compute_supervoxels(tissue, c0, n_segments=300)
    graph = build_supervoxel_graph(labels, tissue, D, c0, c_followup, voxel_size=voxel_spacing_mm)
    if graph["node_centroids"].shape[0] < 5:
        return None, "too few supervoxels after SLIC (degenerate mask?)"
    return graph, "ok"


In [ ]:
lumiere_graphs = {}
lumiere_skipped = []
for i, patient_dir in enumerate(lumiere_patients):
    pid = os.path.basename(patient_dir)
    try:
        graph, note = build_lumiere_graph(patient_dir, verbose=(i == 0))
    except Exception as e:
        graph, note = None, f"exception: {e}"
    if graph is not None:
        lumiere_graphs[pid] = graph
    else:
        lumiere_skipped.append((pid, note))
    if i == 0:
        tag = f"{graph['node_centroids'].shape[0]} supervoxels" if graph else note
        print(f"[{pid}] {tag}")

print(f"\nBuilt {len(lumiere_graphs)} / {len(lumiere_patients)} usable LUMIERE patient graphs.")
if lumiere_skipped:
    print("Skipped patients (reason):")
    for pid, note in lumiere_skipped[:15]:
        print(f"  {pid}: {note}")
    if len(lumiere_skipped) > 15:
        print(f"  ... and {len(lumiere_skipped) - 15} more")

lumiere_real_pair_loaded = len(lumiere_graphs) > 0
# kept for compatibility with the Section 6 forward-pass smoke test below, which
# expects a single demo graph named graph_lum
if lumiere_graphs:
    graph_lum = next(iter(lumiere_graphs.values()))


In [ ]:
# ---- dense (pre-supervoxel) baseline/follow-up pairs, for the CNN/U-Net
# baseline below (Section 4.2 "live" row) -- reuses the exact same loading
# helpers as build_rhuh_graph()/build_lumiere_graph() above so the CNN sees
# the identical baseline/follow-up pair per patient, just without the SLIC
# supervoxel aggregation step. Restricted to the same patient IDs already in
# rhuh_graphs/lumiere_graphs so folds/train-val splits line up exactly.

def build_rhuh_dense(row_pre, row_fu):
    _, c0_2d = seg_and_density_from_row(row_pre)
    _, c_followup_2d = seg_and_density_from_row(row_fu)
    if c0_2d.shape != c_followup_2d.shape:
        return None
    if c0_2d.sum() < 5 or c_followup_2d.sum() < 5:
        return None
    return c0_2d.astype(float), c_followup_2d.astype(float)


def build_lumiere_dense(patient_dir):
    timepoints = find_lumiere_timepoint_dirs(patient_dir)
    if len(timepoints) < 2:
        return None
    seg_baseline = find_lumiere_seg(timepoints[0])
    seg_followup = find_lumiere_seg(timepoints[-1])
    if not (seg_baseline and seg_followup):
        return None
    import nibabel as nib
    seg_b = nib.load(seg_baseline).get_fdata()
    seg_f = nib.load(seg_followup).get_fdata()
    if seg_b.shape != seg_f.shape:
        return None
    c0 = (seg_b > 0).astype(float)
    c_followup = (seg_f > 0).astype(float)
    if c0.sum() < 5 or c_followup.sum() < 5:
        return None
    return c0, c_followup


rhuh_dense = {}
for pid in rhuh_graphs:
    row_pre, row_fu, _ = patient_pairs[pid]
    pair = build_rhuh_dense(row_pre, row_fu)
    if pair is not None:
        rhuh_dense[pid] = pair
print(f"Built {len(rhuh_dense)} / {len(rhuh_graphs)} dense RHUH-GBM pairs for the CNN baseline "
      f"(should match exactly -- same loader inputs, no SLIC-specific skip reasons apply).")

lumiere_dense = {}
for patient_dir in lumiere_patients:
    pid = os.path.basename(patient_dir)
    if pid not in lumiere_graphs:
        continue
    pair = build_lumiere_dense(patient_dir)
    if pair is not None:
        lumiere_dense[pid] = pair
print(f"Built {len(lumiere_dense)} / {len(lumiere_graphs)} dense LUMIERE pairs for the CNN baseline.")


## 6. Test C — PI-GNN forward pass on real graphs

Untrained weights -- this only checks that real data flows through `GliomaGrowthPIGNN.forward()` end-to-end with correct shapes/dtypes (the point of Sections 4-5 was getting real `x`, `edge_index`, `edge_attr`, `c0`, `c_followup` from real anatomy; this section is just plumbing, not a trained result).


In [ ]:
# dt is fixed at 0.1 inside ReactionDiffusionStep, and the placeholder isotropic
# tensor + raw-pixel-unit centroid spacing (voxel_size=1.0, uncalibrated to real mm --
# see Section 0/5 notes) make the physics-derived diffusion conductance w_ij tiny
# (~1e-4 in this cohort). At only 10 steps the model can barely move c from c0 at all,
# regardless of what the learned correction does -- 40 steps gives ~4x the simulated
# time for both diffusion and the reaction term (which compounds near tumor boundaries)
# to have a visible effect, without meaningfully changing runtime (graphs are tiny).
N_STEPS_DEFAULT = 40

def graph_dict_to_tensors(graph, device=DEVICE):
    x = torch.tensor(graph["node_features"], dtype=torch.float, device=device)
    edge_index = torch.tensor(graph["edge_index"], dtype=torch.long, device=device)
    edge_attr = torch.tensor(graph["edge_attr"], dtype=torch.float, device=device)
    c0 = torch.tensor(graph["c0"], dtype=torch.float, device=device)
    c_followup = torch.tensor(graph["c_followup"], dtype=torch.float, device=device)
    boundary_mask = torch.tensor(graph["boundary_mask"], dtype=torch.bool, device=device)
    return x, edge_index, edge_attr, c0, c_followup, boundary_mask

real_graphs = {}
if "graph_rhuh" in dir():
    real_graphs["RHUH-GBM (2D)"] = graph_rhuh
if "graph_lum" in dir() and lumiere_real_pair_loaded:
    real_graphs["LUMIERE (longitudinal)"] = graph_lum

print("Real graphs available for the forward-pass smoke test:", list(real_graphs.keys()))

for name, graph in real_graphs.items():
    x, edge_index, edge_attr, c0, c_followup, boundary_mask = graph_dict_to_tensors(graph)
    model = GliomaGrowthPIGNN(node_feat_dim=x.shape[1], edge_feat_dim=edge_attr.shape[1]).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        trajectory, rhos = model(c0, x, edge_index, edge_attr, n_steps=N_STEPS_DEFAULT)
    elapsed = time.time() - t0

    loss, components = total_loss(trajectory, c_followup, x, edge_index, edge_attr,
                                   model.step, boundary_mask)

    print(f"\n[{name}]")
    print(f"  nodes={x.shape[0]}  edges={edge_index.shape[1]}  node_feat_dim={x.shape[1]}  edge_feat_dim={edge_attr.shape[1]}")
    print(f"  forward pass ({N_STEPS_DEFAULT} steps, untrained): {elapsed*1000:.1f} ms on {DEVICE}")
    print(f"  trajectory shape: {tuple(trajectory.shape)}  (T+1, N)")
    print(f"  total_loss={loss.item():.4f}  components={components}")


## 7. Quick training sanity check (overfit on real graphs)

`train_epoch()` in `pignn_glioma.py` has never been run against real data before -- Sections 4-6 only checked
that a single *untrained* forward pass produces correctly-shaped tensors. This section actually calls
`loss.backward()` / `optimizer.step()` and checks that the loss goes down, which is the first real test of
whether gradients flow correctly through `propagate()`, the edge/rho MLPs, and `total_loss()` end-to-end.

**What this does and doesn't prove:** each real graph here is a single patient (RHUH-GBM: one 2D slice;
LUMIERE: one longitudinal pair), so this trains and evaluates on the *same* graph -- a classic overfitting
setup, deliberately. The point is only to confirm optimization mechanics work (loss decreases, no NaNs, no
shape errors under `backward()`), not to demonstrate generalization -- that needs the multi-patient
`DataLoader`-based run flagged as next steps in Section 8's item 1.


In [ ]:
def run_overfit_check(name, graph, n_epochs=150, n_steps=40, lr=1e-2, seed=0,
                       lambda_prior=0.05, warmup_frac=0.3):
    torch.manual_seed(seed)
    x, edge_index, edge_attr, c0, c_followup, boundary_mask = graph_dict_to_tensors(graph)
    model = GliomaGrowthPIGNN(node_feat_dim=x.shape[1], edge_feat_dim=edge_attr.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"total": [], "data": [], "physics_prior": [], "bc": []}
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        # Staged warmup: pure data-fit for the first warmup_frac of training (lets the model
        # actually move away from the trivial gate=1/rho=0 point using only the Dice signal --
        # confirmed on a real run that a CONSTANT lambda_prior, even at 0.05, still froze data
        # loss completely flat from epoch 30 to 150), then turn the physics-prior term on for
        # the remainder so the final weights are still anchored to the DTI-derived physics.
        current_lambda_prior = 0.0 if epoch < warmup_frac * n_epochs else lambda_prior
        trajectory, _ = model(c0, x, edge_index, edge_attr, n_steps=n_steps)
        loss, components = total_loss(trajectory, c_followup, x, edge_index, edge_attr,
                                       model.step, boundary_mask, lambdas=(1.0, current_lambda_prior, 0.1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        history["total"].append(loss.item())
        for k in ("data", "physics_prior", "bc"):
            history[k].append(components[k])

        if epoch == 0 or (epoch + 1) % max(1, n_epochs // 5) == 0:
            print(f"[{name}] epoch {epoch+1:4d}/{n_epochs}  total={loss.item():.4f}  "
                  f"data={components['data']:.4f}  physics_prior={components['physics_prior']:.4f}  "
                  f"bc={components['bc']:.4f}")

    with torch.no_grad():
        trajectory_final, _ = model(c0, x, edge_index, edge_attr, n_steps=n_steps)
        _, final_components = total_loss(trajectory_final, c_followup, x, edge_index, edge_attr,
                                          model.step, boundary_mask, lambdas=(1.0, lambda_prior, 0.1))
    print(f"[{name}] dice_loss(pred, c_followup): {history['data'][0]:.4f} -> "
          f"{final_components['data']:.4f}  (lower is better; 0 = perfect overlap)")
    return model, history

overfit_results = {}
for name, graph in real_graphs.items():
    print(f"\n{'='*70}\nOverfit sanity check: {name}\n{'='*70}")
    _, history = run_overfit_check(name, graph)
    overfit_results[name] = history


In [ ]:
if overfit_results:
    fig, axes = plt.subplots(1, len(overfit_results), figsize=(6 * len(overfit_results), 4))
    if len(overfit_results) == 1:
        axes = [axes]
    for ax, (name, history) in zip(axes, overfit_results.items()):
        ax.plot(history["total"], label="total")
        ax.plot(history["data"], label="data (dice_loss)")
        ax.plot(history["physics_prior"], label="physics_prior")
        ax.plot(history["bc"], label="bc")
        ax.set_title(name)
        ax.set_xlabel("epoch")
        ax.set_ylabel("loss")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No real graphs available to run the overfit check on -- check Sections 4-5 above.")


## 8. Multi-patient training, FEM baseline, and ablation (toward real Table 2 numbers)

Everything above (Sections 4-7) validates plumbing on one demo graph per dataset. This section runs the
actual experiment Table 2 needs: patient-level k-fold cross-validation on RHUH-GBM, external validation on
LUMIERE using a model trained only on RHUH-GBM, a classical-FEM baseline (identical finite-volume
discretization, no learned correction, one global hand-fit rho -- the Swanson-style comparator from
Section 4.2), and the physics-informed-vs-plain-GNN ablation also from Section 4.2.

**Scope note:** GliODIL, the PINN baseline, and the CNN/U-Net baseline are not reproduced here. Given the
timeline, cite their numbers from the original papers with an explicit non-identical-cohort caveat rather
than risking a rushed, possibly-buggy reproduction. The FEM baseline and the ablation are the two
comparisons cheap and safe enough to run live in the time available, and they're also the two that most
directly isolate this paper's actual claimed contribution (physics-derived edges + physics-prior loss).


In [ ]:
# ---- shared evaluation metrics (paper Section 4.3) ----
SWANSON_THRESHOLD = 0.16  # T2/FLAIR-visible margin convention, used for Dice + recurrence coverage

def evaluate_prediction(c_pred, c_followup, threshold=SWANSON_THRESHOLD):
    c_pred = c_pred.detach()
    pred_mask = (c_pred >= threshold).float()
    true_mask = (c_followup >= threshold).float()  # already binary here (real segmentation-derived);
                                                     # threshold applied anyway in case a future dataset
                                                     # passes a continuous density instead of a 0/1 mask
    intersection = (pred_mask * true_mask).sum().item()
    dice = (2 * intersection + 1e-6) / (pred_mask.sum().item() + true_mask.sum().item() + 1e-6)
    coverage = (intersection + 1e-6) / (true_mask.sum().item() + 1e-6)  # recurrence coverage / sensitivity
    rmse = torch.sqrt(torch.mean((c_pred - c_followup) ** 2)).item()
    mae = torch.mean(torch.abs(c_pred - c_followup)).item()
    return {"dice": dice, "coverage": coverage, "rmse": rmse, "mae": mae}


def evaluate_model_on_patients(model, graphs_dict, patient_ids, n_steps=40):
    rows = []
    for pid in patient_ids:
        x, edge_index, edge_attr, c0, c_followup, boundary_mask = graph_dict_to_tensors(graphs_dict[pid])
        t0 = time.time()
        with torch.no_grad():
            trajectory, _ = model(c0, x, edge_index, edge_attr, n_steps=n_steps)
        elapsed = time.time() - t0
        row = evaluate_prediction(trajectory[-1], c_followup)
        row["inference_ms"] = elapsed * 1000
        row["patient_id"] = pid
        rows.append(row)
    return rows


def summarize_rows(rows, keys=("dice", "coverage", "rmse", "mae", "inference_ms")):
    out = {}
    for k in keys:
        vals = np.array([r[k] for r in rows], dtype=float)
        out[k] = (float(vals.mean()), float(vals.std())) if len(vals) else (float("nan"), float("nan"))
    return out


In [ ]:
def train_multi_patient(graphs_dict, patient_ids, n_epochs=60, n_steps=40, lr=1e-2,
                         physics_informed=True, seed=0, verbose_every=0, warmup_frac=0.3):
    """
    Loops over each patient graph individually per epoch rather than batching via
    torch_geometric's DataLoader/Batch. These graphs are tiny (tens to a few hundred
    nodes), so there is no real throughput benefit to batched collation here, and
    looping sidesteps having to validate that collation behaves correctly for our
    non-standard per-graph fields (c0, c_followup, boundary_mask, n_steps) -- which
    cannot be tested in the offline sandbox this notebook was drafted in.

    physics_informed=False runs the plain data-driven-GNN ablation described in the
    manuscript's 4.2: the physics-derived base conductance column of edge_attr is
    zeroed (so the learned edge gate has no physics prior to modulate) and the
    physics-prior term is dropped from the loss (lambda_prior=0) -- "no PDE residual
    loss and no physics-derived edge weights."
    """
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    sample_graph = graphs_dict[patient_ids[0]]
    node_feat_dim = sample_graph["node_features"].shape[1]
    edge_feat_dim = sample_graph["edge_attr"].shape[1]
    model = GliomaGrowthPIGNN(node_feat_dim=node_feat_dim, edge_feat_dim=edge_feat_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # Annealed down from the synthetic-pretraining-scale 0.5 (see manuscript 3.4), AND staged:
    # a real run showed even a constant 0.05 froze the data-fit term completely flat from early
    # training onward, because the prior's gradient path is far more direct than the data term's
    # (which backpropagates through the full unrolled trajectory). Pure data-fit for the first
    # warmup_frac of epochs lets the model move away from the trivial gate=1/rho=0 point before
    # the physics-prior term is switched on to anchor the remaining training.
    target_lambda_prior = 0.05 if physics_informed else 0.0

    history = []
    for epoch in range(n_epochs):
        current_lambda_prior = 0.0 if epoch < warmup_frac * n_epochs else target_lambda_prior
        lambdas = (1.0, current_lambda_prior, 0.1)
        order = rng.permutation(len(patient_ids))
        epoch_losses = []
        for idx in order:
            pid = patient_ids[idx]
            x, edge_index, edge_attr, c0, c_followup, boundary_mask = graph_dict_to_tensors(graphs_dict[pid])
            if not physics_informed:
                edge_attr = edge_attr.clone()
                edge_attr[:, 0] = 0.0

            optimizer.zero_grad()
            trajectory, _ = model(c0, x, edge_index, edge_attr, n_steps=n_steps)
            loss, _ = total_loss(trajectory, c_followup, x, edge_index, edge_attr,
                                  model.step, boundary_mask, lambdas=lambdas)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        history.append(float(np.mean(epoch_losses)))
        if verbose_every and (epoch == 0 or (epoch + 1) % verbose_every == 0):
            print(f"    epoch {epoch+1:4d}/{n_epochs}  mean_loss={history[-1]:.4f}")

    return model, history


In [ ]:
class FEMBaseline:
    """
    Callable matching GliomaGrowthPIGNN's forward signature -- (c0, node_features,
    edge_index, edge_attr, n_steps) -> (trajectory, None) -- so
    evaluate_model_on_patients() can score it identically to the learned model.

    Implements the classical Swanson-style comparator from the manuscript's 4.2:
    the SAME finite-volume discretization (same edge_attr[:, 0] physics-derived
    conductance, same explicit-Euler update as ReactionDiffusionStep) but with NO
    learned correction -- the gate is fixed at 1 (trust the physics weight exactly)
    and rho is a single global scalar fit once across the training cohort, not a
    per-patient/per-node learned quantity.
    """
    def __init__(self, rho, dt=0.1):
        self.rho = rho
        self.dt = dt

    def __call__(self, c0, node_features, edge_index, edge_attr, n_steps):
        c = c0.clone()
        src, dst = edge_index
        w_base = edge_attr[:, 0]
        trajectory = [c]
        for _ in range(n_steps):
            diffusion = torch.zeros_like(c)
            diffusion.scatter_add_(0, dst, w_base * (c[src] - c[dst]))
            reaction = self.rho * c * (1 - c)
            c = (c + self.dt * (diffusion + reaction)).clamp(0.0, 1.0)
            trajectory.append(c)
        return torch.stack(trajectory, dim=0), None


def fit_global_rho(graphs_dict, patient_ids, rho_candidates, n_steps=40, dt=0.1):
    best_rho, best_mean_loss = None, float("inf")
    for rho in rho_candidates:
        fem = FEMBaseline(rho, dt=dt)
        losses = []
        for pid in patient_ids:
            x, edge_index, edge_attr, c0, c_followup, boundary_mask = graph_dict_to_tensors(graphs_dict[pid])
            trajectory, _ = fem(c0, x, edge_index, edge_attr, n_steps=n_steps)
            losses.append(dice_loss(trajectory[-1], c_followup).item())
        mean_loss = float(np.mean(losses)) if losses else float("inf")
        if mean_loss < best_mean_loss:
            best_mean_loss, best_rho = mean_loss, rho
    return best_rho, best_mean_loss


In [ ]:
# ---- CNN/U-Net autoregressive rollout baseline (paper Section 4.2, live) ----
# Operates on the dense baseline/follow-up grid directly (not the supervoxel
# graph), downsampled to a fixed voxel budget so a 3D-conv rollout stays
# tractable on CPU within Kaggle's free-tier session limits. This is the
# baseline that justifies the graph representation itself, independent of the
# physics contribution: same autoregressive-rollout idea as the ablation GNN
# (no physics prior, pure data-fit), but with local 3x3x3 convolutions on a
# regular grid instead of message passing over an anatomy-aware mesh.
#
# 2D RHUH-GBM slices are treated as 3D with a singleton depth axis (same
# convention as build_rhuh_graph() above), so one architecture and training
# loop covers both cohorts; a singleton axis is never downsampled further
# (see _resize_to_budget below), and Conv3d with kernel_size=3, padding=1
# leaves a size-1 axis at size 1, so it degenerates cleanly to a 2D conv
# there without any special-casing in the model itself.

CNN_VOXEL_BUDGET = 32768  # ~32**3 -- keeps per-patient rollout to a few seconds on CPU

def _resize_to_budget(arr, voxel_budget=CNN_VOXEL_BUDGET):
    if arr.ndim == 2:
        arr = arr[:, :, None]
    shape = np.array(arr.shape, dtype=float)
    free_axes = shape > 1
    if not free_axes.any():
        return np.clip(arr, 0.0, 1.0)
    fixed_product = np.prod(shape[~free_axes]) if (~free_axes).any() else 1.0
    free_shape_product = np.prod(shape[free_axes])
    target_free_product = voxel_budget / fixed_product
    n_free = int(free_axes.sum())
    s = min(1.0, (target_free_product / free_shape_product) ** (1.0 / n_free))
    zoom_factors = np.where(free_axes, s, 1.0)
    out = ndimage.zoom(arr, zoom_factors, order=1)
    return np.clip(out, 0.0, 1.0)


class CNNRollout(nn.Module):
    """Small 3D CNN predicting a per-step density update, unrolled n_steps
    times. dc is scaled by 1/n_steps so total displacement is roughly
    step-count invariant, matching the dt-scaling convention used by
    ReactionDiffusionStep elsewhere in this notebook."""

    def __init__(self, hidden=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(1, hidden, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(hidden, hidden, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(hidden, 1, kernel_size=3, padding=1),
        )

    def forward(self, c0_grid, n_steps=40):
        # c0_grid: (1, 1, D, H, W)
        c = c0_grid
        for _ in range(n_steps):
            dc = self.net(c) / n_steps
            c = torch.clamp(c + dc, 0.0, 1.0)
        return c


def _dense_to_tensor(arr):
    arr_ds = _resize_to_budget(arr)
    return torch.tensor(arr_ds, dtype=torch.float32, device=DEVICE)[None, None]


def _match_shape(arr, target_shape):
    """scipy.ndimage.zoom rounds output size to the nearest int per axis, so
    zooming down and back up does not always land on exactly the original
    shape (off-by-one voxel is common). Crop or zero-pad each axis to force
    an exact match before the elementwise Dice/RMSE computation below, which
    requires identical shapes."""
    out = arr
    for axis, target in enumerate(target_shape):
        cur = out.shape[axis]
        if cur > target:
            out = np.take(out, range(target), axis=axis)
        elif cur < target:
            pad_width = [(0, 0)] * out.ndim
            pad_width[axis] = (0, target - cur)
            out = np.pad(out, pad_width, mode="constant", constant_values=0.0)
    return out


def train_cnn_multi_patient(dense_dict, patient_ids, n_epochs=60, n_steps=40, lr=1e-2, seed=0):
    """Mirrors train_multi_patient()'s per-patient loop and training budget
    (Section 7), but with CNNRollout instead of GliomaGrowthPIGNN and a plain
    soft-Dice data loss only -- no physics-prior term applies to a baseline
    that has no physics-derived edges in the first place."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = CNNRollout().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    ids = [pid for pid in patient_ids if pid in dense_dict]
    history = []
    for epoch in range(n_epochs):
        order = rng.permutation(len(ids))
        epoch_losses = []
        for idx in order:
            pid = ids[idx]
            c0, c_followup = dense_dict[pid]
            c0_t = _dense_to_tensor(c0)
            cf_t = _dense_to_tensor(c_followup)[0, 0]

            optimizer.zero_grad()
            pred = model(c0_t, n_steps=n_steps)[0, 0]
            loss = dice_loss(pred, cf_t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_losses.append(loss.item())
        history.append(float(np.mean(epoch_losses)) if epoch_losses else float("nan"))

    return model, history


def evaluate_cnn_on_patients(model, dense_dict, patient_ids, n_steps=40):
    """Evaluated at native (pre-downsample) resolution: the CNN's downsampled
    prediction is upsampled back to the original baseline/follow-up grid
    shape before computing Dice/coverage/RMSE, so the reported metric is
    against the same follow-up array used elsewhere -- NOT against the
    downsampled grid. This is not identical in resolution to the supervoxel-
    node-level metric used for FEM/ablation/PI-GNN (that comparison is at
    ~150-300 supervoxels/patient); it is the most direct way to score this
    baseline against the ground-truth scan without also having to define a
    matching supervoxel aggregation for a voxel-grid model. See Section 4.2/
    Limitations for this caveat.
    """
    model.eval()
    rows = []
    for pid in patient_ids:
        if pid not in dense_dict:
            continue
        c0, c_followup_native = dense_dict[pid]
        c0_t = _dense_to_tensor(c0)
        t0 = time.time()
        with torch.no_grad():
            pred_ds = model(c0_t, n_steps=n_steps)[0, 0].cpu().numpy()
        native_shape = c_followup_native.shape if c_followup_native.ndim == 3 else c_followup_native.shape + (1,)
        zoom_back = [native_shape[i] / pred_ds.shape[i] for i in range(3)]
        pred_native = np.clip(ndimage.zoom(pred_ds, zoom_back, order=1), 0.0, 1.0)
        pred_native = _match_shape(pred_native, native_shape)  # guard against zoom rounding mismatches
        elapsed = time.time() - t0

        cf_flat = c_followup_native.reshape(native_shape)
        pred_t = torch.tensor(pred_native, dtype=torch.float32)
        cf_t = torch.tensor(cf_flat, dtype=torch.float32)
        row = evaluate_prediction(pred_t, cf_t)
        row["inference_ms"] = elapsed * 1000
        row["patient_id"] = pid
        rows.append(row)
    return rows


In [ ]:
# ---- per-patient inverse-fit personalization experiment (paper Section 3.5) ----
# inverse_fit_patient() previously optimized rho_scale against a loss that never
# depended on it (a bug, now fixed in pignn_glioma.py -- see that file's docstring).
# This cell actually exercises the fixed routine: for each held-out validation
# patient, freeze the fold's already-trained PI-GNN and fit a single scalar
# rho_scale from that patient's own baseline+follow-up scan, then compare against
# (a) the same model's zero-shot prediction (already computed as rows_pi in the CV
# loop below) and (b) a per-patient FEM fit -- the classical iterative alternative
# this routine is meant to be a fast replacement for. All three see the exact same
# patients, so accuracy AND fitting time are directly comparable.

def evaluate_personalized_pi(model, graphs_dict, patient_ids, n_steps=40, n_iters=200, lr=1e-2):
    rows = []
    for pid in patient_ids:
        x, edge_index, edge_attr, c0, c_followup, boundary_mask = graph_dict_to_tensors(graphs_dict[pid])

        t0 = time.time()
        rho_scale = inverse_fit_patient(model, x, edge_index, edge_attr, c0, c_followup,
                                         boundary_mask, n_steps=n_steps, n_iters=n_iters, lr=lr)
        fit_time_ms = (time.time() - t0) * 1000

        t1 = time.time()
        with torch.no_grad():
            trajectory, _ = model(c0, x, edge_index, edge_attr, n_steps=n_steps, rho_scale=rho_scale)
        inference_ms = (time.time() - t1) * 1000

        row = evaluate_prediction(trajectory[-1], c_followup)
        row["fit_time_ms"] = fit_time_ms
        row["inference_ms"] = inference_ms
        row["rho_scale"] = rho_scale
        row["patient_id"] = pid
        rows.append(row)
    return rows


def fit_fem_per_patient_timed(graphs_dict, pid, rho_candidates, n_steps=40):
    """Same grid search as fit_global_rho(), but restricted to this one
    patient's own baseline+follow-up (a genuine per-patient FEM fit, not the
    single fold-wide global rho used for the FEM row in Table 2), and timed
    the same way evaluate_personalized_pi() times inverse_fit_patient()."""
    t0 = time.time()
    best_rho, _ = fit_global_rho(graphs_dict, [pid], rho_candidates, n_steps=n_steps)
    fit_time_ms = (time.time() - t0) * 1000

    fem = FEMBaseline(best_rho)
    x, edge_index, edge_attr, c0, c_followup, boundary_mask = graph_dict_to_tensors(graphs_dict[pid])
    t1 = time.time()
    with torch.no_grad():
        trajectory, _ = fem(c0, x, edge_index, edge_attr, n_steps=n_steps)
    inference_ms = (time.time() - t1) * 1000

    row = evaluate_prediction(trajectory[-1], c_followup)
    row["fit_time_ms"] = fit_time_ms
    row["inference_ms"] = inference_ms
    row["rho"] = best_rho
    row["patient_id"] = pid
    return row


def evaluate_fem_per_patient(graphs_dict, patient_ids, rho_candidates, n_steps=40):
    return [fit_fem_per_patient_timed(graphs_dict, pid, rho_candidates, n_steps=n_steps)
            for pid in patient_ids]


In [ ]:
N_EPOCHS = 60          # small but real training budget -- graphs are tiny (100s of nodes),
                        # fast even on CPU, so this is minutes not hours on Kaggle's free tier
N_STEPS = 40
RHO_CANDIDATES = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]
# widened after a real run picked 0.5 (the old ceiling) in every fold -- that's the search
# hitting its boundary, not necessarily the true optimum, so extend further out.

fold_results = {"proposed": [], "ablation": [], "fem": [], "cnn": [],
                 "personalized_pi": [], "fem_per_patient": []}

for fold_i in range(N_FOLDS):
    val_ids = rhuh_folds[fold_i]
    train_ids = [pid for j, f in enumerate(rhuh_folds) if j != fold_i for pid in f]
    if not val_ids or not train_ids:
        print(f"fold {fold_i}: skipped (empty train or val split -- too few usable patients for {N_FOLDS}-fold CV)")
        continue

    print(f"\n{'='*70}\nFold {fold_i}: train={len(train_ids)} patients, val={len(val_ids)} patients\n{'='*70}")

    model_pi, _ = train_multi_patient(rhuh_graphs, train_ids, n_epochs=N_EPOCHS, n_steps=N_STEPS,
                                       physics_informed=True, seed=fold_i)
    rows_pi = evaluate_model_on_patients(model_pi, rhuh_graphs, val_ids, n_steps=N_STEPS)
    fold_results["proposed"].extend(rows_pi)
    print(f"  proposed (physics-informed): {summarize_rows(rows_pi)}")

    # Personalization experiment (Section 3.5): freeze this fold's already-trained
    # model_pi and fit a per-patient rho_scale on each val patient's own scan, vs.
    # a per-patient FEM fit on the same patients. model_pi is not reused for
    # anything else in this fold after this point, so freezing it here is safe.
    rows_personalized = evaluate_personalized_pi(model_pi, rhuh_graphs, val_ids, n_steps=N_STEPS)
    fold_results["personalized_pi"].extend(rows_personalized)
    print(f"  personalized PI-GNN (per-patient rho_scale fit): {summarize_rows(rows_personalized, keys=('dice','coverage','rmse','mae','inference_ms','fit_time_ms'))}")

    rows_fem_pp = evaluate_fem_per_patient(rhuh_graphs, val_ids, RHO_CANDIDATES, n_steps=N_STEPS)
    fold_results["fem_per_patient"].extend(rows_fem_pp)
    print(f"  FEM (per-patient rho fit): {summarize_rows(rows_fem_pp, keys=('dice','coverage','rmse','mae','inference_ms','fit_time_ms'))}")

    model_ab, _ = train_multi_patient(rhuh_graphs, train_ids, n_epochs=N_EPOCHS, n_steps=N_STEPS,
                                       physics_informed=False, seed=fold_i)
    rows_ab = evaluate_model_on_patients(model_ab, rhuh_graphs, val_ids, n_steps=N_STEPS)
    fold_results["ablation"].extend(rows_ab)
    print(f"  ablation (plain data-driven GNN): {summarize_rows(rows_ab)}")

    best_rho, _ = fit_global_rho(rhuh_graphs, train_ids, RHO_CANDIDATES, n_steps=N_STEPS)
    if best_rho == max(RHO_CANDIDATES):
        print(f"  WARNING: fit_global_rho picked {best_rho}, the largest candidate -- the search "
              f"likely hit its ceiling rather than finding a true optimum. Widen RHO_CANDIDATES above.")
    fem_model = FEMBaseline(best_rho)
    rows_fem = evaluate_model_on_patients(fem_model, rhuh_graphs, val_ids, n_steps=N_STEPS)
    fold_results["fem"].extend(rows_fem)
    print(f"  FEM baseline (global rho={best_rho}): {summarize_rows(rows_fem)}")

    train_ids_dense = [pid for pid in train_ids if pid in rhuh_dense]
    val_ids_dense = [pid for pid in val_ids if pid in rhuh_dense]
    if train_ids_dense and val_ids_dense:
        model_cnn, _ = train_cnn_multi_patient(rhuh_dense, train_ids_dense, n_epochs=N_EPOCHS,
                                                n_steps=N_STEPS, seed=fold_i)
        rows_cnn = evaluate_cnn_on_patients(model_cnn, rhuh_dense, val_ids_dense, n_steps=N_STEPS)
        fold_results["cnn"].extend(rows_cnn)
        print(f"  CNN/U-Net rollout (voxel-grid, no physics): {summarize_rows(rows_cnn)}")
    else:
        print("  CNN/U-Net rollout: skipped this fold (no dense pairs for train or val split)")

print(f"\n{'='*70}\nRHUH-GBM {N_FOLDS}-fold CV summary (pooled across folds\' held-out patients)\n{'='*70}")
for method, rows in fold_results.items():
    print(f"{method:10s}: n={len(rows):3d}  {summarize_rows(rows)}")


In [ ]:
if not RHUH_PATIENT_IDS:
    print("No usable RHUH-GBM patients -- cannot train a final model or run LUMIERE "
          "external validation. Check Section 4's skip-reason printout above.")
    lumiere_results = {}
else:
    print(f"\n{'='*70}\nTraining final models on ALL {N_RHUH} RHUH-GBM patients for LUMIERE external validation\n{'='*70}")
    final_model_pi, _ = train_multi_patient(rhuh_graphs, RHUH_PATIENT_IDS, n_epochs=N_EPOCHS, n_steps=N_STEPS,
                                             physics_informed=True, seed=100)
    final_model_ab, _ = train_multi_patient(rhuh_graphs, RHUH_PATIENT_IDS, n_epochs=N_EPOCHS, n_steps=N_STEPS,
                                             physics_informed=False, seed=100)
    final_rho, _ = fit_global_rho(rhuh_graphs, RHUH_PATIENT_IDS, RHO_CANDIDATES, n_steps=N_STEPS)
    if final_rho == max(RHO_CANDIDATES):
        print(f"WARNING: fit_global_rho picked {final_rho}, the largest candidate -- widen RHO_CANDIDATES.")
    final_fem = FEMBaseline(final_rho)
    print(f"Final global rho (fit on all RHUH-GBM): {final_rho}")

    RHUH_IDS_DENSE = [pid for pid in RHUH_PATIENT_IDS if pid in rhuh_dense]
    final_model_cnn = None
    if RHUH_IDS_DENSE:
        final_model_cnn, _ = train_cnn_multi_patient(rhuh_dense, RHUH_IDS_DENSE, n_epochs=N_EPOCHS,
                                                       n_steps=N_STEPS, seed=100)

    LUMIERE_PATIENT_IDS = sorted(lumiere_graphs.keys())
    lumiere_results = {}
    if LUMIERE_PATIENT_IDS:
        lumiere_results["proposed"] = evaluate_model_on_patients(final_model_pi, lumiere_graphs, LUMIERE_PATIENT_IDS, n_steps=N_STEPS)
        lumiere_results["ablation"] = evaluate_model_on_patients(final_model_ab, lumiere_graphs, LUMIERE_PATIENT_IDS, n_steps=N_STEPS)
        lumiere_results["fem"] = evaluate_model_on_patients(final_fem, lumiere_graphs, LUMIERE_PATIENT_IDS, n_steps=N_STEPS)
        if final_model_cnn is not None:
            lumiere_ids_dense = [pid for pid in LUMIERE_PATIENT_IDS if pid in lumiere_dense]
            lumiere_results["cnn"] = evaluate_cnn_on_patients(final_model_cnn, lumiere_dense, lumiere_ids_dense, n_steps=N_STEPS)
        else:
            lumiere_results["cnn"] = []
        print(f"\nLUMIERE external validation (n={len(LUMIERE_PATIENT_IDS)} patients, model trained only on RHUH-GBM):")
        for method, rows in lumiere_results.items():
            print(f"  {method:10s}: {summarize_rows(rows)}")
    else:
        print("No usable LUMIERE graphs available -- check Section 5 above.")


In [ ]:
def fmt(mean_std):
    mean, std = mean_std
    return f"{mean:.3f} +/- {std:.3f}"

print(f"{'Method':<12}{'Cohort':<14}{'n':<5}{'Dice':<16}{'Coverage':<16}{'RMSE':<16}{'Inference (ms)':<16}")
for method, rows in fold_results.items():
    if not rows:
        continue
    s = summarize_rows(rows)
    print(f"{method:<12}{'RHUH-GBM CV':<14}{len(rows):<5}{fmt(s['dice']):<16}{fmt(s['coverage']):<16}{fmt(s['rmse']):<16}{fmt(s['inference_ms']):<16}")
for method, rows in lumiere_results.items():
    if not rows:
        continue
    s = summarize_rows(rows)
    print(f"{method:<12}{'LUMIERE ext.':<14}{len(rows):<5}{fmt(s['dice']):<16}{fmt(s['coverage']):<16}{fmt(s['rmse']):<16}{fmt(s['inference_ms']):<16}")

print("\nCopy these mean +/- SD numbers into the manuscript\'s Table 2. GliODIL/PINN rows are cited")
print("from their original papers (not reproduced here) -- see Section 4.2's scope note above.")

# ---- paired significance tests, bootstrap CIs, effect sizes, multiple-comparison
# correction (paper Section 4.4 -- promised there but not actually computed until now) ----
try:
    from scipy.stats import wilcoxon, rankdata

    def paired_by_patient(rows_a, rows_b, key="dice"):
        a = {r["patient_id"]: r[key] for r in rows_a}
        b = {r["patient_id"]: r[key] for r in rows_b}
        common = sorted(set(a) & set(b))
        return np.array([a[p] for p in common]), np.array([b[p] for p in common]), common

    def rank_biserial(va, vb):
        # matched-pairs rank-biserial correlation (King & Minium convention):
        # r = (W+ - W-) / (W+ + W-), using ranks of |difference| with zero-diff
        # pairs dropped, consistent with scipy.stats.wilcoxon's own default handling.
        d = va - vb
        d = d[d != 0]
        if len(d) == 0:
            return float("nan")
        ranks = rankdata(np.abs(d))
        w_plus = ranks[d > 0].sum()
        w_minus = ranks[d < 0].sum()
        return float((w_plus - w_minus) / (w_plus + w_minus))

    def bootstrap_ci(values, n_boot=1000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        values = np.asarray(values, dtype=float)
        n = len(values)
        if n == 0:
            return (float("nan"), float("nan"))
        boot_means = np.array([values[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
        lo, hi = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        return float(lo), float(hi)

    print("\nBootstrap 95% CIs on mean recurrence coverage (1000 resamples, RHUH-GBM CV):")
    for method, rows in fold_results.items():
        if not rows:
            continue
        cov = [r["coverage"] for r in rows]
        lo, hi = bootstrap_ci(cov)
        print(f"  {method:<10s}: mean={np.mean(cov):.3f}  95% CI=[{lo:.3f}, {hi:.3f}]  (n={len(cov)})")

    print("\nPaired significance tests (RHUH-GBM CV, Dice, Wilcoxon signed-rank),")
    print("with rank-biserial effect size and Holm-Bonferroni correction across baselines:")
    baseline_names = [m for m in ("ablation", "fem", "cnn") if fold_results.get(m)]
    raw_results = []
    for baseline_name in baseline_names:
        va, vb, common = paired_by_patient(fold_results["proposed"], fold_results[baseline_name], key="dice")
        if len(common) >= 5:
            stat, p = wilcoxon(va, vb)
            r_rb = rank_biserial(va, vb)
            raw_results.append((baseline_name, len(common), p, float(np.mean(va - vb)), r_rb))
        else:
            print(f"  proposed vs {baseline_name}: only {len(common)} paired patients -- too few for a signed-rank "
                  f"test, report descriptively (mean +/- SD above) only.")

    # Holm step-down correction across the baselines actually tested above
    if raw_results:
        order = np.argsort([r[2] for r in raw_results])
        m = len(raw_results)
        adjusted = [None] * m
        running_max = 0.0
        for rank, idx in enumerate(order):
            p_raw = raw_results[idx][2]
            p_adj = min(1.0, p_raw * (m - rank))
            running_max = max(running_max, p_adj)
            adjusted[idx] = running_max
        for (baseline_name, n_common, p_raw, mean_diff, r_rb), p_holm in zip(raw_results, adjusted):
            sig = "*" if p_holm < 0.05 else " "
            print(f"  proposed vs {baseline_name:<9s}: n={n_common:<3d} p={p_raw:.4f}  "
                  f"p_holm={p_holm:.4f}{sig}  mean diff={mean_diff:+.4f}  rank-biserial r={r_rb:+.3f}")
        print("  (* = significant at Holm-corrected alpha=0.05; none significant here means treat")
        print("   proposed/baseline differences as comparable, not as an established advantage.)")
    print("\nPersonalization experiment (Section 3.5): per-patient rho_scale fit vs. zero-shot "
          "PI-GNN vs. a per-patient FEM fit, on the same RHUH-GBM CV validation patients:")
    for method in ("proposed", "personalized_pi", "fem_per_patient"):
        rows = fold_results.get(method, [])
        if not rows:
            continue
        s = summarize_rows(rows, keys=("dice", "coverage", "rmse"))
        fit_times = [r["fit_time_ms"] for r in rows if "fit_time_ms" in r]
        fit_time_str = (f"fit_time={np.mean(fit_times):.1f}+/-{np.std(fit_times):.1f} ms"
                         if fit_times else "fit_time=0 ms (zero-shot, no per-patient fitting)")
        print(f"  {method:<16s}: n={len(rows):<3d} dice={fmt(s['dice'])}  coverage={fmt(s['coverage'])}  "
              f"rmse={fmt(s['rmse'])}  {fit_time_str}")

    va, vb, common = paired_by_patient(fold_results["personalized_pi"], fold_results["proposed"], key="dice")
    if len(common) >= 5:
        stat, p = wilcoxon(va, vb)
        r_rb = rank_biserial(va, vb)
        print(f"  personalized vs. zero-shot PI-GNN: n={len(common)}  p={p:.4f}  "
              f"mean diff={float(np.mean(va - vb)):+.4f}  rank-biserial r={r_rb:+.3f}  "
              f"(not included in the Holm correction above -- a different comparison, personalized "
              f"vs. its own zero-shot baseline rather than proposed vs. an independent baseline)")
    else:
        print(f"  personalized vs. zero-shot PI-GNN: only {len(common)} paired patients -- too few for a "
              f"signed-rank test.")
except ImportError:
    print("\nscipy not available in this environment -- skipping the Wilcoxon/bootstrap/effect-size "
          "analysis (should run fine on Kaggle, where scipy is preinstalled).")


In [ ]:
# ---- qualitative case example (for a manuscript figure) ----
# Picks one RHUH-GBM patient (final model trained on all RHUH-GBM patients,
# same final_model_pi used for the LUMIERE external validation above) and
# saves a baseline / PI-GNN-predicted / actual-follow-up panel to
# case_example.png. Uses a scatter plot over supervoxel centroids rather than
# rasterizing back to a dense grid -- simpler and avoids introducing a new
# resampling step just for a figure. RHUH-GBM is a 2D slice in this pipeline
# (see Section 4/build_rhuh_graph above), so only the first two centroid
# coordinates are used; this cell is RHUH-GBM-specific for that reason.
#
# Download case_example.png after running this on Kaggle and send it back
# for insertion into the manuscript as a qualitative results figure.

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def plot_case_example(pid, graphs_dict, model, n_steps=N_STEPS, save_path="case_example.png"):
    x, edge_index, edge_attr, c0, c_followup, boundary_mask = graph_dict_to_tensors(graphs_dict[pid])
    model.eval()
    with torch.no_grad():
        trajectory, _ = model(c0, x, edge_index, edge_attr, n_steps=n_steps)
    pred = trajectory[-1]
    metrics = evaluate_prediction(pred, c_followup)

    centroids = graphs_dict[pid]["node_centroids"][:, :2]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    panels = [
        (c0.cpu().numpy(), "Baseline (t=0)"),
        (pred.cpu().numpy(), "PI-GNN predicted"),
        (c_followup.cpu().numpy(), "Actual follow-up"),
    ]
    sc = None
    for ax, (values, title) in zip(axes, panels):
        sc = ax.scatter(centroids[:, 1], -centroids[:, 0], c=values, cmap="inferno", vmin=0, vmax=1, s=40)
        ax.set_title(title, fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal")
    fig.colorbar(sc, ax=axes, shrink=0.7, label="tumor cell density")
    fig.suptitle(f"Case example: patient {pid}  (Dice={metrics['dice']:.3f}, "
                 f"coverage={metrics['coverage']:.3f})")
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {save_path} (Dice={metrics['dice']:.3f}, coverage={metrics['coverage']:.3f}, "
          f"RMSE={metrics['rmse']:.3f}) -- download this file and send it back for insertion "
          f"into the manuscript as a qualitative results figure.")


if RHUH_PATIENT_IDS:
    example_pid = RHUH_PATIENT_IDS[0]
    plot_case_example(example_pid, rhuh_graphs, final_model_pi, save_path="case_example.png")
else:
    print("No RHUH-GBM patients available for a case-example figure.")


## 9. Summary & next steps

Section 8 runs real k-fold cross-validated training on RHUH-GBM, external validation on LUMIERE, a live
classical-FEM baseline, a physics-informed vs. plain-data-GNN ablation, a live CNN/U-Net
autoregressive-rollout baseline on a downsampled voxel grid, and now a per-patient personalization
experiment (Section 3.5) -- alongside bootstrap 95% CIs, rank-biserial effect sizes, and
Holm-Bonferroni-corrected paired significance tests (Section 4.4's promises, actually computed now).

- **Real segmentation -> graph, multi-patient** (RHUH-GBM) and **real longitudinal baseline/follow-up,
  multi-patient** (LUMIERE) replace the single-demo-patient checks from earlier notebook versions.
- **Section 8's k-fold CV** gives patient-level Dice, recurrence coverage, RMSE, and inference time for the
  proposed method, the plain-data-GNN ablation, the classical-FEM baseline, and the CNN/U-Net rollout --
  with Wilcoxon signed-rank tests, bootstrap CIs, rank-biserial effect sizes, and Holm correction across
  baselines wherever enough paired patients are available.
- **LUMIERE external validation** reuses models trained only on RHUH-GBM (including the CNN), matching the
  manuscript's stated design (LUMIERE held out entirely from training/tuning).
- **A qualitative case-example figure** (one RHUH-GBM patient's baseline/predicted/actual-follow-up panel)
  is generated at the end of Section 8 for direct insertion into the manuscript.
- **`inverse_fit_patient()`'s rho_scale threading bug is fixed AND now exercised**: each fold's frozen
  model_pi gets a per-patient rho_scale fit on its own held-out validation patients
  (`evaluate_personalized_pi()`), compared against zero-shot PI-GNN and a genuine per-patient FEM fit
  (`evaluate_fem_per_patient()`, as opposed to the single fold-wide global rho used for the Table 2 FEM
  row) on the same patients -- both accuracy and fitting time are reported, since fast personalization
  vs. FEM's slower iterative per-patient search is the actual claim in Section 3.5.
- **Real per-axis voxel spacing (mm) is now used for LUMIERE** (`build_lumiere_graph()` pulls it from the
  NIfTI header via `nib.load(...).header.get_zooms()`), instead of the previous pixel-index default of 1.0
  for every axis. The finite-volume conductance formula divides by physical distance squared, so this
  could change every LUMIERE edge weight -- worth comparing this run's numbers against the previous
  output file's before assuming anything else changed. RHUH-GBM has no NIfTI affine to pull real spacing
  from (2D slices from a HuggingFace image dataset) and still uses pixel-unit spacing; this asymmetry is
  now called out explicitly in code comments and should be in the Limitations section too.

**What this notebook still does *not* do, and what's cited instead:**
1. **GliODIL and PINN are not reproduced here.** Live GliODIL reproduction was assessed and specifically
   set aside (not just skipped by default): the official implementation's own documentation requires a
   single GPU with >18.5 GB memory (30-45 min/patient) against a CPU-only budget here, plus BraTS-convention
   4-class segmentation and white/grey-matter tissue maps this pipeline's RHUH-GBM/LUMIERE preprocessing
   does not produce. Their Table 2 numbers are cited from the original papers instead, with an explicit
   caveat that the test cohort/split isn't identical to this paper's. The CNN/U-Net baseline, by contrast,
   needed no external framework and is run live above.
2. **UPenn-GBM is not tested.** The Kaggle mirror attached in an earlier run (`sauravilalge/upenn-gbm`) is
   the raw per-visit clinical DICOM archive, with no segmentation and no DTI/FA/ADC maps -- it can't drive a
   real-anatomy FEM run or a real diffusion tensor.
3. **Neither RHUH-GBM nor LUMIERE has real per-patient DTI in this pipeline** -- both use the same
   placeholder isotropic diffusion tensor as the synthetic sanity check (Table 3). This remains the most
   consequential open item: it means the anisotropic, physics-informed edge weighting central to the
   method's motivation has not actually been tested against real diffusion data, which is a plausible
   contributor to the non-significant Dice differences in Table 2. The voxel-spacing fix above corrects the
   *distance* term in the conductance formula, not this *tensor* term -- both matter, independently.
4. Target volume ratio (Section 4.3's third primary metric) is not computed here -- it needs a
   coverage-matched comparison against a fixed clinical margin (20mm/15mm) and physical (mm) node
   coordinates. Dice and recurrence coverage, the other two primary metrics, are computed in Section 8.
5. `A_ij` (shared boundary area) is still the constant placeholder from `build_supervoxel_graph()`.
6. LUMIERE's baseline/follow-up scans are compared as-is; a shape mismatch (flagged by `build_lumiere_graph`
   returning "needs registration") means that patient is silently skipped rather than misaligned.
7. Everything new in this pass (the voxel-spacing fix, the personalization experiment) was written and
   reasoned through without a local torch/torch_geometric/scipy/skimage environment to test against --
   expect to need at least one debugging pass on Kaggle before these numbers are submission-ready. Read
   every print statement Section 8 produces (skip counts, fold sizes, chosen rho, the personalization
   fit-time/accuracy comparison) before trusting the final table.
